# PS2 — Device + Serial Grain Cascade Analysis (v1)
CUBIC MARS Chicago / CTA-Ventra · built 21-Jul-2026 · revised 21-Jul-2026 (Phase-4 elimination)

**Scope.** PS2 is descriptive/analytical only — there is no trained model, no SageMaker endpoint,
no promotion gate. This notebook extends the device/city-grain `PS2_enhanced.ipynb` (31 charts,
14-Jul) down to **component/serial grain**.

**IMPORTANT — revised 21-Jul-2026.** This notebook originally assumed a "Phase-4" gold layer
(`device_ps2_component_day`, `device_ps2_serial_catalog`, `device_ps2_assessment`,
`device_ps2_markov_matrix`, `device_cross_ps`, `serial_cross_ps`) had already been built in
Databricks. A live-catalog check (`information_schema.tables`, exact + fuzzy search across every
schema) confirmed **none of these six tables exist** — there are only **5 real Gold tables**
(`device_ps1_daily`, `device_ps2_chains`, `device_ps3_incident`, `device_ps4_hourly`,
`device_ps5_component`). Per PK's direction, this notebook now derives everything it needs
directly from those 5 real tables instead:
- **Serial grain** comes from `device_ps2_chains.DEVICE_SERIAL_NUMBER`, which is present on every
  row — no separate component/serial table is needed.
- **Cross-PS cascade attribution** is built inline from `device_ps1_daily`, `device_ps3_incident`,
  and `device_ps4_hourly` (defensive column resolution + printed resolution, not a precomputed
  spine table).
- **Serial age** (used for age-stratified velocity and chronic-cascade recurrence) is resolved from
  `silver.hw_config_current` when it is available and its serials match at a reasonable rate; when
  it doesn't, both families fall back to a clearly-labeled proxy (days since a serial's earliest
  observed cascade in `device_ps2_chains`) and print which source applies on every run.

**Grain policy (do not deviate without updating this cell):**
- Re-run **natively at serial grain**: phi-correlation, Markov transitions (self-transition-rate
  fallback), HMM 3-state regime, device/serial recurrence, business-impact scoring.
- Re-run at serial grain **with an explicit minimum-support floor**, suppressed-count reported,
  never silently dropped: conditional probability P(B\|A,T), FP-growth association rules.
- **Device/subsystem/facility grain only** — serial grain destroys the signal, do not rerun:
  network centrality, facility contagion, ignition/termination, cascade velocity/windows
  (stratified by *current* serial age via `SERIAL_AGE_DAYS` instead of a full per-serial rerun).
- **New families** (beyond the reference set): chronic-cascade recurrence (serial, explicitly
  descriptive/non-modeled — see the disclaimer in that section, this is NOT a PS5 survival model),
  cross-PS cascade attribution (built inline as above, correlational only, never re-derives PS4's
  anomaly flags), subsystem-pair lead/lag timing distributions, cascade cost/severity Sankey source
  table, and a device/serial→ServiceNow CMDB `cmdb_ci` sys_id map.

**Output contract.** Every table this notebook produces is written as **Parquet + a `manifest.json`**
to S3 under `s3://<export-bucket>/chicago/ps2_outputs/<table_name>/computed_date=<date>/run_id=<uuid>/`, with
mandatory `grain`, `run_id`, `computed_date`, `notebook_version` columns on every row. A new
S3-triggered Lambda (`cubic-mars-ps2-rds-push`) consumes the manifest, auto-evolves the RDS schema
(`ADD COLUMN IF NOT EXISTS` only — never a destructive `ALTER TYPE`), and loads the table. This
replaces the old copy/paste "generate INSERT SQL by hand" pattern for everything this notebook
produces; the 15 existing device-grain `ps2_*` tables from `PS2_enhanced.ipynb` keep using their
existing backfill-SQL path unchanged.

In [ ]:
# ── CAPTURE CELL : Run this FIRST before any other cell ─────────────────────
# All print() output appears in BOTH the cell output AND PS2_pavan/console.log
# All plt.show() figures appear in BOTH the cell output AND PS2_pavan/<name>.png
#
# NOTE (21-Jul-2026): OUT_DIR renamed from "_PS2_serialgrain_Outputs" to
# "PS2_pavan" per PK's request. Every table written via write_output() below
# now also lands under PS2_pavan/<TVM|Gates|Validators|Cross_Device>/
# <device_grain|serial_grain|...>/<table>.csv as a human-browsable local copy,
# in addition to the unchanged S3 Parquet + manifest.json path that
# cubic-mars-ps2-rds-push watches. See the Configuration and Generic output
# writer cells for the device-type resolution and split logic.
import os, sys
OUT_DIR = "PS2_pavan"
os.makedirs(OUT_DIR, exist_ok=True)

class _Tee:
    """Writes to Jupyter cell output + log file simultaneously."""
    def __init__(self, jupyter_stream, log_fh):
        self._jupyter = jupyter_stream
        self._log     = log_fh
        self._is_tee  = True

    def write(self, data):
        try:
            self._jupyter.write(data); self._jupyter.flush()
        except Exception:
            pass
        try:
            self._log.write(data); self._log.flush()
        except Exception:
            pass

    def flush(self):
        try: self._jupyter.flush()
        except Exception: pass
        try: self._log.flush()
        except Exception: pass

    def __getattr__(self, name):
        return getattr(self._jupyter, name)

if not getattr(sys.stdout, "_is_tee", False):
    _log_fh = open(os.path.join(OUT_DIR, "console.log"), "w", buffering=1, encoding="utf-8")
    sys.stdout = _Tee(sys.stdout, _log_fh)
    sys.stderr = _Tee(sys.stderr, _log_fh)

print("[capture] console.log active ->", os.path.join(OUT_DIR, "console.log"))

In [ ]:
# ── PREFLIGHT : numpy sanity gate (same as PS2_enhanced.ipynb — copy verbatim) ─
def _numpy_preflight():
    try:
        import numpy as np
        v = np.__version__
        major = int(v.split(".")[0])
        _ = np.array([1.0, 2.0]).sum()
        if major >= 2:
            print(f"[preflight] WARNING numpy=={v} (2.x) in a live kernel.")
            print("[preflight] Run the install cell, then Kernel -> Restart Kernel -> Run-All.")
            return False
        print(f"[preflight] numpy=={v}  OK (multiarray loads cleanly).")
        return True
    except Exception as e:
        print("[preflight] numpy is BROKEN in this kernel:", str(e)[:140])
        return False

NUMPY_OK = _numpy_preflight()

In [ ]:
# ── CELL : Install dependencies (numpy-2.x-safe, install-only-if-missing) ──────
import subprocess, sys, importlib

# NOTE (21-Jul-2026, 2nd fix): the previous version of this cell always ran
# `pip install` with hard upper-bound pins on the WHOLE stack (pandas<2.3,
# pyarrow<17, sagemaker<3.0, ...). On this Studio image those pins are stale —
# other preinstalled Studio components (sagemaker-studio, sagemaker-mlops/
# serve/train) already need NEWER pandas/pyarrow/sagemaker-core than our pins
# allowed, so pip's resolver downgraded them to satisfy our upper bounds,
# breaking those other components (visible as the wall of "requires X, but
# you have Y which is incompatible" errors). We do not need to pin packages
# that are already present and importable — the ONLY packages this notebook
# is missing are hmmlearn and mlxtend (per the ModuleNotFoundError). Fixed:
# for everything except numpy, only install if the import actually fails, and
# with no upper-bound version cap, so pip is free to pick whatever is
# compatible with what this Studio image already has on disk.

# NOTE (21-Jul-2026, 3rd fix): that "no upper-bound pin" design had a real gap
# -- installing a genuinely-missing package (hmmlearn/mlxtend) with no cap
# lets pip's resolver pull in whatever numpy/pandas/pyarrow/scikit-learn THAT
# package's own setup wants, and pip is happy to silently UPGRADE an
# already-imported package to satisfy it (that's what produced "you have
# numpy 2.5.1" here even though the numpy preflight/branch below never ran a
# numpy install itself). Fixed by writing a pip CONSTRAINTS file that locks
# every core compiled library this kernel already has imported (numpy, scipy,
# pandas, pyarrow, scikit-learn) to its CURRENT live version, passed to every
# pip call via `-c`.

# NOTE (21-Jul-2026, 4th fix): the constraints file alone was NOT enough. PK's
# next run showed numpy correctly held at 1.26.4 (the 3rd fix's ABI-hot-swap
# protection worked) but pip still installed `mlxtend==0.25.0`, which declares
# `numpy>=2.3.5` -- a version it cannot possibly get from a kernel pinned to
# numpy==1.26.4. In other words, `-c constraints.txt` stopped pip from
# upgrading an already-loaded library, but did NOT stop pip from installing a
# NEW package whose own declared requirement the pinned library can't satisfy
# -- pip's post-install audit reports that mismatch, but the install itself
# is not blocked by `-c` alone. Fixed two ways: (a) every pinned library is
# now ALSO passed as a direct, explicit requirement on the SAME pip install
# command line (not just inside the constraints file) -- this forces pip to
# solve for the pinned library and the new package TOGETHER in one resolution,
# so an incompatible candidate (like mlxtend 0.25.0) is rejected and pip
# either backtracks to a compatible release or fails loudly, instead of
# installing the incompatible one and only flagging it afterward; (b) after
# any install pip reports as successful, this cell now runs a FRESH
# subprocess import check using the *exact* import statements the Imports
# cell further down actually uses -- this is ground truth (does the import
# really work?) rather than trusting pip's metadata-only conflict report,
# which is exactly the kind of check that would have caught the mlxtend
# situation immediately instead of leaving it as an ambiguous warning wall.

_PIN_CANDIDATES = ["numpy", "scipy", "pandas", "pyarrow", "sklearn"]
_PIN_PIP_NAMES = {"sklearn": "scikit-learn"}
_LIVE_VERSIONS_BEFORE = {}
for _mod_name in _PIN_CANDIDATES:
    try:
        _m = importlib.import_module(_mod_name)
        _LIVE_VERSIONS_BEFORE[_mod_name] = getattr(_m, "__version__", None)
    except ImportError:
        _LIVE_VERSIONS_BEFORE[_mod_name] = None

_CONSTRAINTS_PATH = "/tmp/_ps2_pin_constraints.txt"
_PINNED_SPECS = []   # e.g. ["numpy==1.26.4", "pandas==2.3.3", ...] -- used BOTH as the
                      # constraints-file content AND as direct requirement args (see 4th-fix note)
for _mod_name, _ver in _LIVE_VERSIONS_BEFORE.items():
    if _ver is None:
        continue
    if _mod_name == "numpy" and not NUMPY_OK:
        # numpy is about to be deliberately changed by the branch below -- do NOT
        # pin it here, or we'd be pinning it to the very version we're fixing.
        continue
    _pip_name = _PIN_PIP_NAMES.get(_mod_name, _mod_name)
    _PINNED_SPECS.append(f"{_pip_name}=={_ver}")
with open(_CONSTRAINTS_PATH, "w") as _f:
    _f.write("\n".join(_PINNED_SPECS) + "\n")
print(f"[install] Pinned {len(_PINNED_SPECS)} already-loaded compiled libraries for this kernel "
      f"session (passed as both a pip constraint AND a direct requirement on every install below, "
      f"so pip cannot pick an incompatible version of anything new it installs): {_PINNED_SPECS}")

# Exact import statements the Imports cell further down actually runs, for the two
# genuinely-new packages -- used as ground-truth post-install verification (4th fix).
_EXACT_IMPORT_CHECKS = {
    "hmmlearn": "from hmmlearn import hmm",
    "mlxtend": ("from mlxtend.frequent_patterns import apriori, association_rules\n"
                "from mlxtend.preprocessing import TransactionEncoder"),
}

def _verify_importable(import_name):
    """Runs the EXACT import this notebook actually needs in a fresh subprocess -- ground
    truth for "does this really work", independent of what pip's metadata-only conflict
    checker reports (which can miss or misreport real usability, as it did for mlxtend)."""
    check_code = _EXACT_IMPORT_CHECKS.get(import_name, f"import {import_name}")
    result = subprocess.run([sys.executable, "-c", check_code], capture_output=True, text=True)
    if result.returncode == 0:
        print(f"[install] {import_name:<14} verified importable in a fresh subprocess "
              f"(matches the exact import this notebook's Imports cell will run) — OK.")
        return True
    print(f"[install] {import_name:<14} *** INSTALLED BUT THE REAL IMPORT FAILS *** -- pip "
          f"reported success, but this exact import raised an error (pip's metadata-only "
          f"conflict report can miss this). Traceback:\n{result.stderr[-2000:]}")
    return False


def _ensure(import_name, pip_spec=None, min_version=None):
    """Install pip_spec (defaults to import_name) ONLY if import_name cannot
    already be imported -- never touches an already-working package. Every
    install pins the already-loaded compiled libraries BOTH as a constraint
    file AND as direct requirement args on the same command line, so pip
    solves for compatibility up front instead of installing something
    incompatible and reporting the conflict afterward. A successful pip exit
    code is then double-checked with a real import in a fresh subprocess."""
    pip_spec = pip_spec or import_name
    try:
        mod = importlib.import_module(import_name)
        print(f"[install] {import_name:<14} already present (v{getattr(mod, '__version__', '?')}) — left as-is.")
    except ImportError:
        spec = f"{pip_spec}>={min_version}" if min_version else pip_spec
        print(f"[install] {import_name:<14} MISSING — installing '{spec}' (no upper-bound pin on "
              f"{pip_spec} itself, but {len(_PINNED_SPECS)} already-loaded libraries pinned directly "
              f"on this same command) ...")
        cmd = [sys.executable, "-m", "pip", "install", "-q", "-c", _CONSTRAINTS_PATH, spec] + _PINNED_SPECS
        result = subprocess.run(cmd, check=False, capture_output=True, text=True)
        if result.returncode != 0:
            print(f"[install] {import_name:<14} FAILED under the pinned-requirement install -- this "
                  f"means '{spec}' genuinely needs a newer version of an already-loaded library "
                  f"than this kernel has (a real incompatibility, not something to silently "
                  f"paper over). pip's resolver output:\n{result.stderr[-2000:]}")
        else:
            print(f"[install] {import_name:<14} pip reported success under the pinned requirements — "
                  f"verifying the real import now...")
            _verify_importable(import_name)

# numpy is the one package this notebook truly needs to control the version
# of (hmmlearn/scikit-learn C-extension ABI compatibility) -- guarded by the
# preflight cell above, unchanged from the original design.
if NUMPY_OK:
    print("[install] numpy already OK in this kernel — skipping numpy/scipy pip install "
          "to avoid a live-process ABI hot-swap.")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy>=1.26,<2", "scipy>=1.11,<1.14"], check=False)
    print("[install] Installed numpy<2 + matching scipy. You MUST Kernel -> Restart Kernel -> "
          "Run All now -- a C-extension cannot be hot-swapped in a running process.")

# Everything else: install-only-if-missing, no upper-bound pin on the package itself, so we
# never fight this image's own already-resolved versions of packages it ships with -- but
# every already-loaded compiled library is pinned directly on each install command (see the
# 4th-fix note above), so pip cannot silently pick an incompatible version of anything new.
_ensure("pandas")
_ensure("pyarrow")
_ensure("s3fs")
_ensure("sklearn", pip_spec="scikit-learn")
_ensure("matplotlib")
_ensure("seaborn")
_ensure("networkx")
_ensure("hmmlearn", min_version="0.3")
_ensure("mlxtend", min_version="0.23")
_ensure("boto3")
_ensure("sagemaker")
_ensure("sagemaker_mlflow", pip_spec="sagemaker-mlflow")
_ensure("mlflow", min_version="2.13")

# Safety net -- even with the pinned installs above, confirm none of the already-loaded
# compiled libraries actually changed version. If one did, warn loudly and specifically.
_RESTART_NEEDED = False
for _mod_name, _before in _LIVE_VERSIONS_BEFORE.items():
    if _before is None:
        continue
    try:
        _after = getattr(importlib.import_module(_mod_name), "__version__", None)
    except ImportError:
        _after = None
    if _after != _before:
        _RESTART_NEEDED = True
        print(f"[install] *** {_mod_name} changed version DURING this cell: {_before} -> {_after}. "
              f"This is exactly the live-kernel ABI-hot-swap risk the preflight cell warns about.")

if _RESTART_NEEDED:
    print("[install] >>> Kernel -> Restart Kernel -> Run All is REQUIRED before continuing -- "
          "one or more already-loaded compiled libraries changed version above. <<<")
else:
    print("[install] Done. No already-loaded compiled library changed version — no kernel "
          "restart required unless the numpy branch above fired (it prints its own instruction). "
          "If anything above shows 'MISSING — installing', re-run the Imports cell next. "
          "If anything shows '*** INSTALLED BUT THE REAL IMPORT FAILS ***', that package's "
          "install must be fixed (a compatible version pinned) before the Imports cell will work "
          "-- share that traceback rather than proceeding past it.\n"
          "NOTE: pip's 'dependency resolver' warning wall (autogluon-*, sparkmagic, skops, ...) "
          "reports the FULL environment's pre-existing conflicts on every install, most of which "
          "belong to Studio-bundled packages this notebook never imports -- only a conflict "
          "naming one of hmmlearn/mlxtend/pandas/pyarrow/scikit-learn/numpy/scipy actually matters "
          "here, and this cell's own checks above (not that warning wall) are the reliable signal "
          "for whether this notebook is actually broken.")

In [ ]:
# ── CELL : Imports ──────────────────────────────────────────────────────────
import os, json, time, uuid, warnings, datetime
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as pa_ds
import pyarrow.fs as pa_fs
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import networkx as nx
import boto3, sagemaker
import scipy.stats as stats

from collections import Counter, defaultdict
from itertools import combinations

from hmmlearn import hmm
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

import mlflow

# NEW (21-Jul-2026, 5th fix): the previous version of this only wrapped the FIRST import
# attempt -- if the fallback (`from sagemaker.session import Session`) also failed, that
# raised uncaught and crashed this whole cell (exactly what PK hit: `ModuleNotFoundError:
# No module named 'sagemaker.session'` on the fallback line itself, meaning this kernel's
# installed 'sagemaker' package doesn't have a working sagemaker.session submodule at all
# -- a non-standard/partial install of that package on this Studio image, not something
# this notebook's own install cell touched, since `sagemaker` was already present and
# _ensure() only installs a package that's missing entirely). Neither `Session` nor
# `get_execution_role` is actually called anywhere else in this notebook except to define
# SM_SESSION/ROLE in the Configuration cell below (grep-confirmed) -- this notebook never
# trains or deploys anything (analysis-only, no endpoint, no model registry entry), so a
# failure here should degrade gracefully, not crash the run. Both attempts are now wrapped,
# and a diagnostic dump (sagemaker.__file__/__version__/dir()) prints on failure so the
# root cause is visible without needing to open a shell on the kernel.
try:
    from sagemaker import Session
except ImportError:
    try:
        from sagemaker.session import Session
    except ImportError as _e:
        print(f"[imports] WARNING: could not import Session from sagemaker or "
              f"sagemaker.session ({type(_e).__name__}: {_e}). SM_SESSION will be None in "
              f"the Configuration cell below -- nothing else in this notebook calls it, "
              f"so this is safe to continue past.")
        Session = None
        try:
            import sagemaker as _sm_diag
            print(f"[imports]   sagemaker.__file__    = {getattr(_sm_diag, '__file__', '?')}")
            print(f"[imports]   sagemaker.__version__ = {getattr(_sm_diag, '__version__', '?')}")
            print(f"[imports]   dir(sagemaker) (first 40) = {sorted(dir(_sm_diag))[:40]}")
        except Exception:
            pass

try:
    from sagemaker import get_execution_role
except ImportError:
    try:
        from sagemaker.session import get_execution_role
    except ImportError as _e:
        print(f"[imports] WARNING: could not import get_execution_role from sagemaker or "
              f"sagemaker.session ({type(_e).__name__}: {_e}). ROLE will be None in the "
              f"Configuration cell below -- nothing else in this notebook calls it, so "
              f"this is safe to continue past.")
        get_execution_role = None

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
print("Imports OK")

In [ ]:
# ── THEME : CUBIC MARS elegant identity (copied verbatim from PS2_enhanced.ipynb
#    so serial-grain figures drop into the same weekly deck without recolouring) ─
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap

NAVY, INK, CLOUD, GRIDCLR = "#1E3A5F", "#33475B", "#F4F7FB", "#DCE4EC"
PALETTE = ["#9DC3E6", "#A9D18E", "#F4CE7A", "#F1A9A0", "#C9A9DA", "#8FCFC9", "#B6C7D9", "#E8B7C6"]
SEV_COLORS = {"low": "#A9D18E", "mid": "#F4CE7A", "high": "#F1A9A0"}
SEQ_CMAP  = LinearSegmentedColormap.from_list("mars_seq",  ["#F4F7FB", "#CFE0F1", "#9DC3E6", "#6FA8DC", "#3E7CB1", NAVY])
HEAT_CMAP = LinearSegmentedColormap.from_list("mars_heat", ["#F4F7FB", "#D8ECDD", "#F4CE7A", "#EBA98C", "#D98C7A"])
DIV_CMAP  = LinearSegmentedColormap.from_list("mars_div",  ["#C0504D", "#F1A9A0", "#F4F7FB", "#A9D18E", "#4E8A57"])

def apply_mars_theme():
    sns.set_theme(style="white")
    mpl.rcParams.update({
        "figure.facecolor": "white", "axes.facecolor": "white",
        "axes.edgecolor": GRIDCLR, "axes.labelcolor": INK, "text.color": INK,
        "xtick.color": INK, "ytick.color": INK, "axes.grid": True,
        "grid.color": GRIDCLR, "grid.linewidth": 0.6, "font.size": 10.5,
        "axes.titleweight": "bold", "axes.titlecolor": NAVY,
    })

apply_mars_theme()
print("Theme applied")

In [ ]:
# ── CELL : Configuration ────────────────────────────────────────────────────
# Analysis-only notebook (PS2 has no endpoint / no model registry entry).

ARTIFACT_BUCKET = "cubic-mars-pm-s3-datalake-dev-artifacts-170202974600"
GOLD_BUCKET     = "cubic-mars-pm-s3-datalake-dev-gold-170202974600"
# NEW (21-Jul-2026, real-run fix): bronze tables live in a SEPARATE bucket, not
# under a bronze/ prefix inside GOLD_BUCKET. Confirmed via the project canon:
# S3 medallion buckets are cubic-mars-pm-s3-datalake-dev-{raw,bronze,silver,gold,artifacts}-170202974600.
# The first live run's FileNotFoundError on cta_servicenow_cmdb_ci was this
# bug -- load_gold() was pointed at GOLD_BUCKET for a bronze-layer table.
BRONZE_BUCKET   = "cubic-mars-pm-s3-datalake-dev-bronze-170202974600"
# NEW (21-Jul-2026): silver tables live in their own bucket too (same medallion
# pattern as bronze above) -- used only for the optional hw_config_current
# enrichment below.
SILVER_BUCKET   = "cubic-mars-pm-s3-datalake-dev-silver-170202974600"
EXPORT_BUCKET   = ARTIFACT_BUCKET     # push-Lambda watches this bucket's chicago/ps2_outputs/ prefix
EXPORT_PREFIX   = "chicago/ps2_outputs"

# Existing device/city-grain chain table (unchanged source, reused from PS2_enhanced.ipynb)
GOLD_TABLES = {
    "device_ps2_chains":     "chicago/gold/device_ps2_chains",
    # NEW (21-Jul-2026, real-run finding): the 6 "Phase-4" component/serial-grain gold
    # tables this notebook originally assumed (device_ps2_component_day,
    # device_ps2_serial_catalog, device_ps2_assessment, device_ps2_markov_matrix,
    # device_cross_ps, serial_cross_ps) DO NOT EXIST. Confirmed by running a Databricks
    # discovery notebook (information_schema.tables, exact-name + fuzzy search across
    # every schema in mars_dev): zero matches, anywhere, for any of the six. There are
    # only 5 real gold tables. Serial grain is now derived directly from
    # device_ps2_chains's own DEVICE_SERIAL_NUMBER column (confirmed present on every
    # row in the first live run) instead of a separate table; cross-PS attribution is
    # computed inline from the 4 real PS1/PS3/PS4/PS5 gold tables below rather than
    # from a nonexistent precomputed spine. See the loader cell and the
    # cross_ps_attribution rebuild for how each is used.
    "device_ps1_daily":      "chicago/gold/device_ps1_daily",       # PS1: will_fail_3d/7d/14d
    "device_ps3_incident":   "chicago/gold/device_ps3_incident",    # PS3: failure_level severity
    "device_ps4_hourly":     "chicago/gold/device_ps4_hourly",      # PS4: ensemble_anomaly_flag
    "device_ps5_component":  "chicago/gold/device_ps5_component",   # PS5: COMPONENT_SERIAL_NBR grain
    # CMDB extract for the ServiceNow cmdb_ci map -- this "chicago/bronze/..." prefix
    # was an UNVERIFIED GUESS and is now confirmed WRONG (see CMDB_CONFIRMED_URI below,
    # which is what the loader actually reads). Left here only so this dict continues to
    # document "this table lives in the bronze layer" at a glance; not used for the load.
    "cta_servicenow_cmdb_ci": "chicago/bronze/cta_servicenow_cmdb_ci",
    # NOTE (updated 21-Jul-2026, per PK): the source for actual ServiceNow incident
    # records (e.g. validating cascade attribution against confirmed incidents, if a
    # future family joins them) has changed AGAIN -- Cubic/CTA now publish a single
    # conformed bronze table, `servicenow_incident_conformed`, which replaces BOTH of
    # the two previously-tracked tables: `servicenow_incident` (257,246 rows / 321
    # cols) and `cta_servicenow_incident` (278,001 rows / 44 cols, the older/narrower
    # mirror). Neither predecessor should be used going forward -- use
    # `servicenow_incident_conformed` exclusively. Nothing in this notebook currently
    # reads any of these three tables (this is still forward-looking documentation, not
    # an active code path), so this rename alone does not require a re-run; recorded
    # here so the right table gets picked if/when a ServiceNow-incident-joining family
    # is actually added. This is a SEPARATE table from `cta_servicenow_cmdb_ci` (the
    # CMDB/asset extract used by device_cmdb_map below) -- that one is unaffected by
    # this rename.
}

# hw_config_current (SILVER, not gold) is the only potential source of TRUE
# component-install-age data (component_age_days, device_serial_number,
# COMPONENT_SERIAL_NBR) -- used only as an OPTIONAL enrichment for the
# chronic-cascade-recurrence and age-stratified-velocity families. Its exact
# S3 prefix is UNVERIFIED (same status the Phase-4 tables were in before being
# confirmed missing via the Databricks discovery notebook) -- if this
# FileNotFoundErrors, those two families fall back to a device_ps2_chains-only
# proxy rather than failing; if you want the real age data, confirm this
# table's path the same way (DESCRIBE DETAIL in Databricks) and update it here.
SILVER_TABLES = {
    "hw_config_current": "chicago/silver/hw_config_current",
}

# CONFIRMED real S3 location for cta_servicenow_cmdb_ci (21-Jul-2026, via PK's Databricks
# discovery notebook run: DESCRIBE DETAIL against the live Unity Catalog table). This is a
# Unity-Catalog-MANAGED table, so its physical path is an internal __unitystorage UUID path,
# not the human-readable "chicago/bronze/..." prefix guessed above -- that guess is why the
# first live run's CMDB load 404'd even after the bucket was fixed. Confirmed 42,959 rows.
CMDB_CONFIRMED_URI = (
    "s3://cubic-mars-pm-s3-datalake-dev-bronze-170202974600/"
    "__unitystorage/schemas/dc13c6f9-9a56-432a-a65f-8d58b2f1f1b8/"
    "tables/5a137180-97db-400e-a1bd-eba889191510"
)

# CONFIRMED real S3 location for hw_config_current (24-Jul-2026, via PK's DESCRIBE DETAIL
# against mars_dev.silver.hw_config_current). Same Unity-Catalog-managed pattern as CMDB
# above -- the "chicago/silver/hw_config_current" prefix in SILVER_TABLES was an unverified
# guess and is now confirmed wrong; this is the real physical path. Delta-partitioned by
# city_id (single file, 825KB) -- pd.read_parquet() reads the Hive-style partition directory
# directly and recovers city_id as a normal column, no special handling needed.
# Real schema (DESCRIBE, 24-Jul-2026): DEVICE_ID, COMPONENT_DESCRIPTION, COMPONENT_SERIAL_NBR,
# LAST_REPORTED_DTM, REPORTED_CHANGED_DTM, OPERATOR_ID, FACILITY_ID, hw_source,
# component_age_days, DEVICE_KEY, DEVICE_NAME, DEVICE_TYPE_NAME, mars_device_category,
# FACILITY_NAME, OPERATOR_NAME, device_serial_number, city_id.
HW_CONFIG_CONFIRMED_URI = (
    "s3://cubic-mars-pm-s3-datalake-dev-silver-170202974600/"
    "__unitystorage/schemas/9b61b579-cbfe-457a-b384-a13f544c8793/"
    "tables/8ebee855-4b5f-47d3-8216-a00c80d18ffa"
)

# NEW (21-Jul-2026, 5th fix): SM_SESSION/ROLE are not used anywhere else in this notebook
# (grep-confirmed) -- wrapped so a broken sagemaker SDK install on this kernel (see the
# Imports cell's diagnostic dump above) degrades to None instead of crashing here too.
try:
    SM_SESSION = Session() if Session is not None else None
    if SM_SESSION is None:
        print("[config] SM_SESSION = None (Session import failed above) -- not used "
              "elsewhere in this notebook, continuing without it.")
except Exception as _e:
    print(f"[config] Session() call failed ({type(_e).__name__}: {_e}) -- not used "
          f"elsewhere in this notebook, continuing without it.")
    SM_SESSION = None

try:
    ROLE = get_execution_role() if get_execution_role is not None else None
    if ROLE is None:
        print("[config] ROLE = None (get_execution_role import failed above) -- not used "
              "elsewhere in this notebook, continuing without it.")
except Exception as _e:
    print(f"[config] get_execution_role() call failed ({type(_e).__name__}: {_e}) -- not "
          f"used elsewhere in this notebook, continuing without it.")
    ROLE = None

REGION     = boto3.Session().region_name
SM_CLIENT  = boto3.client("sagemaker", region_name=REGION)
S3_CLIENT  = boto3.client("s3", region_name=REGION)

MLFLOW_SERVER_ARN = "arn:aws:sagemaker:us-east-1:170202974600:mlflow-tracking-server/cubic-mars-mlflow-server-dev"
os.environ["MLFLOW_TRACKING_SERVER_ARN"] = MLFLOW_SERVER_ARN
try:
    MLFLOW_TRACKING_URI = SM_CLIENT.describe_mlflow_tracking_server(
        TrackingServerName="cubic-mars-mlflow-server-dev")["TrackingServerUrl"]
    print(f"MLflow managed server: {MLFLOW_TRACKING_URI}")
except Exception:
    MLFLOW_TRACKING_URI = "file:///home/sagemaker-user/mlruns"
    print("MLflow managed server not found — using local file fallback")
mlflow.set_tracking_uri(MLFLOW_SERVER_ARN)

EXPERIMENT_NAME  = "chicago-ps2-serialgrain-cascade-analysis"
NOTEBOOK_VERSION = "ps2_serialgrain_v1"
RUN_ID           = str(uuid.uuid4())
CDATE            = datetime.date.today().isoformat()
CITY_ID          = "CHI"

TIME_WINDOWS = {
    "0-5 min": (0, 5), "5-15 min": (5, 15), "15-30 min": (15, 30),
    "30-60 min": (30, 60), "60+ min": (60, 9999),
}
DEVICE_CATEGORIES = ["TVM", "GATE", "VALIDATOR"]

# Minimum-support floors — sparsity guardrail for serial-grain conditional-probability
# and association-rule families (Design decision A.1 / Risk 3 in the implementation plan).
MIN_SUPPORT_SERIAL_COND_PROB = 20     # min (subsystem-pair, window, serial) observations
MIN_SUPPORT_SERIAL_ASSOC     = 0.02   # min itemset support fraction at serial grain

mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Run ID           : {RUN_ID}")
print(f"Notebook version : {NOTEBOOK_VERSION}")
print(f"Computed date    : {CDATE}")
print(f"Export target    : s3://{EXPORT_BUCKET}/{EXPORT_PREFIX}/<table>/computed_date={CDATE}/run_id={RUN_ID}/")

# NOTE (21-Jul-2026): if the kernel dies partway through a run, it is most
# likely OOM — this notebook computes several O(n^2)-ish matrices (phi
# correlation, HMM fits, association rules) across the full device_ps2_chains
# + component_day history at BOTH device and serial grain. Before a full
# re-run, check the kernel's instance type (top-right of the notebook, next
# to the kernel name in SageMaker Studio) and prefer a memory-optimized size
# (e.g. ml.r5.2xlarge / ml.m5.4xlarge) over a small default. write_output()
# now prints peak RSS after every family so a rising trend is visible before
# it becomes a crash, not just after.
print("If the kernel dies again: check the Studio kernel instance type (top-right of this "
      "notebook) and prefer a memory-optimized size (ml.r5.2xlarge / ml.m5.4xlarge) before "
      "Restart Kernel and Run All — watch the 'peak RSS so far' line write_output() now "
      "prints after every family.")

In [ ]:
# ── CELL : Generic Gold-table loader (schema-introspecting, defensive) ────────
# Reads each table as Parquet on S3 and PRINTS the live column list before any
# downstream cell assumes a column exists — this is the load-time equivalent of
# the `if col in df.columns` guards used throughout this notebook. If a table's
# real schema differs from what a downstream cell assumes, this cell tells you
# in one place instead of a cryptic KeyError three sections later.
def load_gold(table_key, required=True, bucket=None, table_dict=None, full_uri=None):
    # full_uri overrides bucket/prefix entirely -- needed for Unity-Catalog-managed tables
    # whose physical path is an internal __unitystorage UUID path rather than a predictable
    # "<layer>/<schema>/<table>" prefix (see CMDB_CONFIRMED_URI in the Configuration cell).
    if full_uri:
        s3_path = full_uri
    else:
        prefix = (table_dict or GOLD_TABLES)[table_key]
        s3_path = f"s3://{bucket or GOLD_BUCKET}/{prefix}"
    try:
        frame = pd.read_parquet(s3_path, storage_options={"client_kwargs": {"region_name": REGION}})
        print(f"[load] {table_key:<28} {frame.shape} <- {s3_path}")
        print(f"       columns: {list(frame.columns)}")
        return frame
    except Exception as e:
        msg = f"[load] {table_key:<28} FAILED <- {s3_path}  ({type(e).__name__}: {str(e)[:160]})"
        if required:
            print(msg); raise
        print(msg + "  -- optional, continuing without it")
        return None

# NEW (21-Jul-2026, OOM fix): device_ps1_daily/device_ps3_incident/device_ps4_hourly are
# only ever used (cell "Cross-PS cascade attribution") to pull ONE flag-like column each,
# plus DEVICE_ID. Eagerly loading all their columns in full here is what most likely caused
# the kernel death PK hit right after this cell -- device_ps4_hourly in particular is
# HOURLY grain and can be very large. Fixed by only reading each table's SCHEMA here (no
# row data at all) via pyarrow.dataset, printing the column list exactly as before for
# visibility, and deferring the actual read to the cross-PS cell -- where it reads ONLY
# the 2 columns it actually needs (id_col + the resolved flag column), not the whole table.
# device_ps5_component is not read by ANY family in this notebook version -- it is not
# loaded at all (not even its schema) to avoid the wasted S3 call; add a peek/column-limited
# load here, following the same pattern, if/when a future family needs it.
def peek_gold_schema(table_key, bucket=None, table_dict=None, full_uri=None):
    """Reads ONLY the Parquet schema (column names, no row data) -- used for large tables
    where only 1-2 columns are actually needed downstream, to avoid materializing the
    whole frame just to find out which columns exist."""
    if full_uri:
        s3_path = full_uri
    else:
        prefix = (table_dict or GOLD_TABLES)[table_key]
        s3_path = f"s3://{bucket or GOLD_BUCKET}/{prefix}"
    try:
        # FIX (24-Jul-2026, live-run bug): pyarrow.dataset.dataset(), when given an EXPLICIT
        # filesystem= object (S3FileSystem here), requires a bare "bucket/key" path -- NOT a
        # "s3://..." URI -- because the filesystem object already encodes the protocol. Passing
        # the full URI raised ArrowInvalid on every call ("Expected an S3 object path of the form
        # 'bucket/key...', got a URI") for device_ps1_daily/device_ps3_incident/device_ps4_hourly,
        # silently leaving *_SCHEMA_COLS = None for all three and degrading both the cross-PS
        # attribution family (cell 43) and the OOS-to-chargeable family (cell 49) without a loud
        # failure. load_gold() above didn't hit this because pd.read_parquet(storage_options=...)
        # goes through s3fs/fsspec, which DOES accept full s3:// URIs -- different library,
        # different convention. Strip the scheme before handing the path to pa_ds.dataset().
        _fs_path = s3_path[len("s3://"):] if s3_path.startswith("s3://") else s3_path
        dataset = pa_ds.dataset(_fs_path, format="parquet", filesystem=pa_fs.S3FileSystem(region=REGION))
        cols = list(dataset.schema.names)
        print(f"[peek] {table_key:<28} {len(cols)} columns (schema only, no row data) <- {s3_path}")
        print(f"       columns: {cols}")
        return cols, s3_path
    except Exception as e:
        print(f"[peek] {table_key:<28} FAILED <- {s3_path}  ({type(e).__name__}: {str(e)[:160]})")
        return None, s3_path


print("Loading Gold tables (device/city grain, full) + peeking PS1/PS3/PS4 schemas only "
      "(column-limited read deferred to the cross-PS cell -- see the OOM-fix note above)...")
df_device  = load_gold("device_ps2_chains")
PS1_SCHEMA_COLS, PS1_URI = peek_gold_schema("device_ps1_daily")     # PS1: will_fail_3d/7d/14d
PS3_SCHEMA_COLS, PS3_URI = peek_gold_schema("device_ps3_incident")  # PS3: failure_level severity
PS4_SCHEMA_COLS, PS4_URI = peek_gold_schema("device_ps4_hourly")    # PS4: ensemble_anomaly_flag (hourly grain -- largest table, hence the peek-only treatment above)
df_cmdb    = load_gold("cta_servicenow_cmdb_ci", required=False, full_uri=CMDB_CONFIRMED_URI)  # bronze, Unity-Catalog-managed path (confirmed 21-Jul); small table (42,959 rows), safe to load in full

# hw_config_current: SILVER, real component-install-age AND component-serial enrichment.
# FIX (24-Jul-2026): path confirmed via DESCRIBE DETAIL (see HW_CONFIG_CONFIRMED_URI in the
# Configuration cell) -- the old SILVER_TABLES guess 404'd on every prior run. This table also
# carries COMPONENT_SERIAL_NBR/device_serial_number -- the client data-foundation doc's
# prescribed serial source (100% populated per the doc, vs dim_device's 91.8% null) -- so this
# load now also feeds serial-identity resolution below, not just serial age.
df_hwconfig = load_gold("hw_config_current", required=False, full_uri=HW_CONFIG_CONFIRMED_URI)

df_device["transit_day"] = pd.to_datetime(df_device["transit_day"])
GOLD_VERSION = None
print(f"\nDevice-grain chains: {df_device.shape}, date range "
      f"{df_device['transit_day'].min().date()} -> {df_device['transit_day'].max().date()}")

# Serial identity column: resolved DIRECTLY off df_device (device_ps2_chains) now --
# confirmed present as DEVICE_SERIAL_NUMBER on every row in the first live run (21-Jul).
# There is no separate component/serial-grain gold table (see the GOLD_TABLES note in
# the Configuration cell), so every "serial grain" family below groups df_device by
# this column instead of loading a second source.
SERIAL_COL_CANDIDATES = ["DEVICE_SERIAL_NUMBER", "COMPONENT_SERIAL_NBR", "device_serial_number", "serial_nbr"]
def find_serial_col(frame):
    if frame is None:
        return None
    for c in SERIAL_COL_CANDIDATES:
        if c in frame.columns:
            return c
    return None

SERIAL_COL = find_serial_col(df_device) or "DEVICE_SERIAL_NUMBER"
print(f"\nSerial identity column resolved to: {SERIAL_COL} (on device_ps2_chains directly)")
if SERIAL_COL in df_device.columns:
    print(f"Distinct serials in device_ps2_chains: {df_device[SERIAL_COL].nunique():,}")

# ── Serial-grain FIX (24-Jul-2026, live-run finding): device_ps2_chains.DEVICE_SERIAL_NUMBER
# was confirmed ~99.99% null on a live run (2,621,305 / 2,621,574 rows), collapsing every
# serial-grain family in this notebook to ~1 distinct group. Traced via the DDL to
# dim_device.DEVICE_SERIAL_NUMBER (only 8.2% populated at the source -- see R2-15 /
# sql/silver/06_dim_device__create.sql, "91.8% NULL confirmed"). bronze.cta_servicenow_cmdb_ci's
# serial_number column is 99.28% populated instead (confirmed live), and its asset_tag column
# was empirically confirmed (tested against real DEVICE_IDs from this run, not assumed) to match
# 96.8% (4,523/4,673) of device_ps2_chains's real DEVICE_IDs. Reusing that same df_cmdb source the
# ServiceNow-CMDB-map cell already loads, enriched here so EVERY serial-grain family benefits
# (SERIAL_COL is read by name downstream, not re-resolved per family).
SERIAL_COL_RESOLVED = "DEVICE_SERIAL_NUMBER_RESOLVED"

if df_cmdb is not None and "asset_tag" in df_cmdb.columns and "serial_number" in df_cmdb.columns:
    cmdb_serial_map = (
        df_cmdb[["asset_tag", "serial_number"]]
        .dropna(subset=["asset_tag", "serial_number"])
        .drop_duplicates(subset=["asset_tag"], keep="first")
    )
    cmdb_serial_map["_device_id_str"] = cmdb_serial_map["asset_tag"].astype(str)
    df_device["_device_id_str"] = df_device["DEVICE_ID"].astype(str)

    df_device = df_device.merge(
        cmdb_serial_map[["_device_id_str", "serial_number"]].rename(columns={"serial_number": "_cmdb_serial"}),
        on="_device_id_str", how="left",
    ).drop(columns=["_device_id_str"])

    # Prefer the CMDB serial; fall back to device_ps2_chains's own (near-empty) column only
    # when CMDB has no match for this device -- never silently drops a device.
    df_device[SERIAL_COL_RESOLVED] = df_device["_cmdb_serial"].fillna(df_device.get(SERIAL_COL))
    df_device = df_device.drop(columns=["_cmdb_serial"])

    n_devices = df_device["DEVICE_ID"].nunique()
    n_resolved = df_device.loc[df_device[SERIAL_COL_RESOLVED].notna(), "DEVICE_ID"].nunique()
    print(f"[serial-fix] CMDB-enriched serial resolved for {n_resolved:,}/{n_devices:,} devices "
          f"({n_resolved/max(n_devices,1):.1%}) via cta_servicenow_cmdb_ci.serial_number "
          f"(join key: asset_tag, empirically confirmed -- see the 24-Jul diagnostic).")
    SERIAL_COL = SERIAL_COL_RESOLVED
else:
    print(f"[serial-fix] SKIPPED -- df_cmdb unavailable or missing asset_tag/serial_number columns; "
          f"falling back to the original (near-empty) {SERIAL_COL} column.")

print(f"\nSerial identity column NOW resolved to: {SERIAL_COL}")
if SERIAL_COL in df_device.columns:
    print(f"Distinct serials (post-fix): {df_device[SERIAL_COL].nunique():,}")

# ── NEW (21-Jul-2026): device-category (type) column detection ────────────────
# PK asked for outputs to be split out per device type (TVM / Gates / Validators)
# in addition to device-id/serial-id grain. The canonical lakehouse column for
# this is `mars_device_category`, present on nearly every silver/gold table
# joined from dim_device (confirmed via the chicago-data-catalog snapshot) — a
# few tables may carry it under a different name, so this resolves defensively
# the same way SERIAL_COL is resolved above, rather than hardcoding one name.
# Expected values match DEVICE_CATEGORIES in the Configuration cell: "TVM",
# "GATE", "VALIDATOR". If a table's real values differ, write_output()'s local
# split below will file unrecognized values under Other_Device_Types/ with a
# printed warning rather than silently mis-filing or dropping rows.
DEVICE_CATEGORY_COL_CANDIDATES = ["mars_device_category", "DEVICE_TYPE_NAME", "device_category", "DEVICE_CATEGORY", "DEVICE_TYPE"]
def find_device_category_col(frame):
    if frame is None:
        return None
    for c in DEVICE_CATEGORY_COL_CANDIDATES:
        if c in frame.columns:
            return c
    return None

DEVICE_CATEGORY_COL = find_device_category_col(df_device)
if DEVICE_CATEGORY_COL:
    print(f"Device-category column resolved to: {DEVICE_CATEGORY_COL}")
else:
    print("Device-category column NOT found on device_ps2_chains -- the per-device-type "
          "(TVM/Gates/Validators) local folder split under PS2_pavan/ will be SKIPPED for "
          "every table (everything still writes to PS2_pavan/Cross_Device/ and to S3/RDS, "
          "fully unaffected). Check the live column list printed above; if the real column "
          "has a different name, add it to DEVICE_CATEGORY_COL_CANDIDATES and re-run this cell.")

# Device-id -> category lookup, re-attached onto device-keyed output tables (see
# write_output()'s local device-type split) right before they're written, since every
# per-family groupby/aggregate naturally drops this column along the way.
DEVICE_ID_TO_CATEGORY = (
    df_device.drop_duplicates("DEVICE_ID").set_index("DEVICE_ID")[DEVICE_CATEGORY_COL].to_dict()
    if DEVICE_CATEGORY_COL and DEVICE_CATEGORY_COL in df_device.columns else {}
)
print(f"Device-id -> category lookup built for {len(DEVICE_ID_TO_CATEGORY):,} devices "
      "(re-attached onto device-keyed output tables just before write_output() so the "
      "TVM/Gates/Validators local folder split has a column to split on).")

# NEW (21-Jul-2026): serial-age lookup for the age-stratified-velocity and
# chronic-cascade-recurrence families. Two sources, tried in order:
#   1. REAL age from hw_config_current (silver) if it loaded and its device-serial
#      values actually match device_ps2_chains's serials at a reasonable rate.
#   2. FALLBACK proxy: days since this serial's earliest observed cascade in
#      device_ps2_chains -- this is a cascade-history span, NOT true install age,
#      and is labeled as such everywhere it is used (never silently presented as real).
SERIAL_AGE_DAYS = {}
SERIAL_AGE_IS_REAL = False
SERIAL_AGE_SOURCE = "none"

if df_hwconfig is not None:
    hw_serial_col = next((c for c in df_hwconfig.columns
                           if c.lower() in ("device_serial_number", "serial_nbr")), None)
    hw_age_col = next((c for c in df_hwconfig.columns if "component_age_days" in c.lower()), None)
    hw_changed_col = next((c for c in df_hwconfig.columns if "changed" in c.lower()), None)
    print(f"hw_config_current column resolution: serial={hw_serial_col}, age_days={hw_age_col}, "
          f"changed_date={hw_changed_col}")
    if hw_serial_col and (hw_age_col or hw_changed_col):
        tmp_hw = df_hwconfig.copy()
        if hw_age_col:
            tmp_hw["_age_days"] = pd.to_numeric(tmp_hw[hw_age_col], errors="coerce")
        else:
            tmp_hw["_age_days"] = (pd.Timestamp(CDATE) - pd.to_datetime(tmp_hw[hw_changed_col], errors="coerce")).dt.days
        # one row per device serial: take the MAX age across its components (oldest
        # component in service under that physical unit) -- a documented choice, not
        # the only valid one; flip to .min() if the intent should be "youngest part".
        age_by_serial = tmp_hw.dropna(subset=["_age_days"]).groupby(hw_serial_col)["_age_days"].max()
        SERIAL_AGE_DAYS = age_by_serial.to_dict()
        device_serials = set(df_device[SERIAL_COL].dropna().astype(str).unique()) if SERIAL_COL in df_device.columns else set()
        matched = len(device_serials & set(str(k) for k in SERIAL_AGE_DAYS))
        match_rate = matched / max(len(device_serials), 1)
        print(f"hw_config_current age lookup: {len(SERIAL_AGE_DAYS):,} serials, "
              f"{matched:,}/{len(device_serials):,} ({match_rate:.1%}) match device_ps2_chains's serials.")
        if match_rate >= 0.05:
            SERIAL_AGE_IS_REAL = True
            SERIAL_AGE_SOURCE = "hw_config_current.component_age_days (real, max across components per serial)"
        else:
            print("Match rate too low to trust -- falling back to the cascade-history-span proxy below.")
            SERIAL_AGE_DAYS = {}
    else:
        print("hw_config_current loaded but could not resolve a serial + age/changed-date column -- "
              "falling back to the cascade-history-span proxy below.")

if not SERIAL_AGE_IS_REAL and SERIAL_COL in df_device.columns:
    first_seen = df_device.groupby(SERIAL_COL)["transit_day"].min()
    SERIAL_AGE_DAYS = {str(k): max((pd.Timestamp(CDATE) - v).days, 1) for k, v in first_seen.items()}
    SERIAL_AGE_SOURCE = ("days since this serial's EARLIEST OBSERVED cascade in device_ps2_chains -- "
                          "a cascade-history-span PROXY, not true install/component age (hw_config_current "
                          "was unavailable or did not match); treat recurrence-rate figures using this as "
                          "directional, not absolute")
    print(f"Serial-age PROXY built for {len(SERIAL_AGE_DAYS):,} serials: {SERIAL_AGE_SOURCE}")

print(f"SERIAL_AGE_IS_REAL = {SERIAL_AGE_IS_REAL}  (source: {SERIAL_AGE_SOURCE})")

# NEW (21-Jul-2026, OOM fix): df_hwconfig / tmp_hw are not referenced anywhere else in this
# notebook once SERIAL_AGE_DAYS is built -- free them now rather than letting a second full
# gold/silver-table frame sit in memory for the rest of the run.
import gc as _gc7
for _name in ("tmp_hw", "df_hwconfig"):
    if _name in globals() and globals()[_name] is not None:
        del globals()[_name]
df_hwconfig = None   # keep the name defined (harmless) in case any later cell checks "is not None"
_gc7.collect()
print("Freed hw_config_current frame(s) from memory after building SERIAL_AGE_DAYS.")

## Helper functions
Reused verbatim from `PS2_enhanced.ipynb` (`parse_chain`, `get_all_subsystems`, `chain_to_binary_flags`,
`phi_corr`) plus new **grain-generic** helpers so every family below takes a `group_col` parameter
(`"DEVICE_ID"` or `SERIAL_COL`) instead of being written twice. This is also what keeps the output
tables consistently shaped for the auto-ALTER push-Lambda (Design A.3 in the implementation plan).

In [ ]:
# ── CELL : Helper functions (reused + grain-generic extensions) ───────────────
import logging

COMMON_TAGS = {
    "ps": "PS2", "city": "chicago", "gold_version": str(GOLD_VERSION),
    "architecture": "CUBIC-MARS-AWS-v7", "analysis_type": "cascade_pattern_analysis_serialgrain",
}

def parse_chain(chain_str, sep="->"):
    if pd.isna(chain_str) or not chain_str:
        return []
    return [s.strip() for s in str(chain_str).split(sep) if s.strip()]

def get_all_subsystems(df_sub, chain_col="subsystem_chain"):
    all_subs = set()
    for chain in df_sub[chain_col].dropna():
        all_subs.update(parse_chain(chain))
    return sorted(all_subs)

def chain_to_binary_flags(df_sub, subsystems, chain_col="subsystem_chain"):
    rows = []
    for chain in df_sub[chain_col]:
        present = set(parse_chain(chain))
        rows.append({s: int(s in present) for s in subsystems})
    return pd.DataFrame(rows, index=df_sub.index)

def phi_corr(x, y):
    n = len(x)
    n11 = (x & y).sum(); n10 = (x & ~y).sum(); n01 = (~x & y).sum(); n00 = (~x & ~y).sum()
    n1_, n0_, n_1, n_0 = n11 + n10, n01 + n00, n11 + n01, n10 + n00
    denom = np.sqrt(n1_ * n0_ * n_1 * n_0)
    return (n * n11 - n1_ * n_1) / denom if denom > 0 else 0.0

def save_and_log(fig, path, run=None):
    fig.savefig(path, dpi=100, bbox_inches="tight")
    plt.show(); plt.close(fig)
    if run is not None:
        mlflow.log_artifact(path)
    return path

# ---- Grain-generic re-run helpers -------------------------------------------

def phi_matrix_for_grain(df_sub, chain_col="subsystem_chain"):
    """Subsystem x subsystem phi-correlation matrix computed over whatever slice
    of df_sub is passed in (device-grain: pass the whole frame; serial-grain: the
    caller pre-filters/groups by SERIAL_COL and calls this per serial, or — more
    usefully — passes the WHOLE frame with an extra 'entity' grouping key handled
    by the caller, since a full serial-by-serial phi matrix would be N_serials
    separate tiny matrices. See CELL: Phi correlation (serial) below for the
    actual aggregation strategy used."""
    subs = get_all_subsystems(df_sub, chain_col)
    if len(subs) < 2:
        return pd.DataFrame(), subs
    flags = chain_to_binary_flags(df_sub, subs, chain_col).astype(bool)
    mat = pd.DataFrame(index=subs, columns=subs, dtype=float)
    for a in subs:
        for b in subs:
            mat.loc[a, b] = 1.0 if a == b else phi_corr(flags[a], flags[b])
    return mat, subs

def markov_matrix_for_chains(chains):
    """chains: list[list[str]] of subsystem sequences. Returns (states, P) transition matrix."""
    chains = [c for c in chains if len(c) >= 2]
    if not chains:
        return [], np.zeros((0, 0))
    states = sorted({s for c in chains for s in c})
    idx = {s: i for i, s in enumerate(states)}
    M = np.zeros((len(states), len(states)))
    for c in chains:
        for a, b in zip(c[:-1], c[1:]):
            M[idx[a], idx[b]] += 1
    rs = M.sum(1, keepdims=True); rs[rs == 0] = 1
    return states, M / rs

def fit_hmm_3state(sequences_numeric):
    """sequences_numeric: 1D array-like, one value per cascade-day, a numeric severity/length
    feature. Fits a 3-state Gaussian HMM (Critical/Moderate/Minor by mean-ordering).

    NEW (21-Jul-2026): standardizes the input before fitting. Raw chain_length spans a
    wide, heavy-tailed range (minor ~2 vs critical ~40+ in the first live run), which
    produces very large-magnitude log-likelihoods (~ -2.6M) where hmmlearn's raw
    (non-relative) convergence-delta check can flag numerically-insignificant jitter near
    the optimum as "Model is not converging" even when the fit itself is stable and the
    resulting states are well-separated and sensible -- exactly what PK's run showed (a
    clean 3-way minor/moderate/critical split, with the warning still firing). Standardizing
    keeps the EM optimization numerically well-conditioned; state means are converted back
    to original units by the caller (mu/sigma are returned for that). Convergence status is
    captured and returned rather than swallowed, so callers can record it as a QA column on
    the output table instead of leaving it as a warning nobody downstream can act on."""
    X_raw = np.asarray(sequences_numeric, dtype=float).reshape(-1, 1)
    if len(X_raw) < 30:
        return None
    mu, sigma = float(X_raw.mean()), float(X_raw.std())
    if sigma == 0:
        return None
    X = (X_raw - mu) / sigma
    model = hmm.GaussianHMM(n_components=3, covariance_type="diag", n_iter=200, tol=1e-4,
                             min_covar=1e-3, random_state=13)
    # NEW (21-Jul-2026, 7th fix): standardizing (previous fix) genuinely improved
    # numerical conditioning -- PK's next run showed the delta shrink from -51.2 to
    # -1.03, a >45x improvement -- but a small residual negative delta can still occur
    # at this data volume purely from floating-point summation noise across hundreds of
    # thousands of per-sample log-likelihood terms (hmmlearn's own source only forgives
    # deltas within sqrt(machine epsilon) =~ 1.5e-8 of zero, an absolute -- not
    # relative -- threshold, so it will never fully absorb noise at a multi-million-
    # magnitude log-likelihood no matter how well-conditioned the fit is). Root-caused
    # precisely why the previous fix's warnings.catch_warnings() didn't silence the
    # printed line: hmmlearn's ConvergenceMonitor.report() emits it via Python's
    # `logging` module (`logging.getLogger("hmmlearn.base").warning(...)`), a
    # completely different mechanism from the `warnings` module -- catching warnings
    # can never intercept a logging call. Fixed by raising that specific logger's
    # threshold above WARNING for the duration of the fit (restored immediately after,
    # so this cannot mask an unrelated ERROR-level message elsewhere). This is purely
    # about console noise -- the actual convergence outcome is still captured
    # faithfully via model.monitor_.converged below and written to the output table's
    # QA column either way, exactly as before.
    _hmm_logger = logging.getLogger("hmmlearn.base")
    _hmm_logger_prev_level = _hmm_logger.level
    _hmm_logger.setLevel(logging.ERROR)
    try:
        with warnings.catch_warnings(record=True):
            warnings.simplefilter("always")
            model.fit(X)
    finally:
        _hmm_logger.setLevel(_hmm_logger_prev_level)
    converged = bool(getattr(model.monitor_, "converged", False))
    n_iter_run = int(getattr(model.monitor_, "iter", 0))
    order = np.argsort(model.means_.flatten())  # ordering is scale-invariant under this affine rescale
    return {"model": model, "order": order, "mu": mu, "sigma": sigma,
            "converged": converged, "n_iter_run": n_iter_run}

def suppressed_floor(n, floor):
    """Returns True if a cell should be SUPPRESSED for low support (report the
    suppression, never silently drop the row)."""
    return n < floor

print("Helper functions ready (grain-generic).")

## Generic S3 output writer (Parquet + manifest.json)
Every analysis section below calls `write_output(frame, family, grain)` exactly once per table it
produces. This is the ONLY place table-naming/columns are enforced, which is what lets the
`cubic-mars-ps2-rds-push` Lambda stay generic instead of needing per-table code.

In [ ]:
# ── CELL : Generic output writer -> S3 Parquet + manifest.json ────────────────
# PLUS (21-Jul-2026) a local, human-browsable copy split by device type
# (TVM / Gates / Validators) x grain, under PS2_pavan/ — added per PK's request.
# IMPORTANT: the S3 Parquet + manifest.json block below is UNCHANGED from the
# original build — it is exactly what cubic-mars-ps2-rds-push (already tested
# against local Postgres: create-table, add-column, type-conflict-skip cases)
# watches. Its key layout and manifest schema must not change. The local
# device-type split is a pure ADDITION for human review; a failure inside it
# is caught and printed as a warning, never allowed to break the S3/RDS path.
import gc, resource

_ALL_MANIFESTS = []

# DEVICE_CATEGORIES (Configuration cell) values -> local folder names, matching
# how PK asked for the split: "gates, validators and TVM".
DEVICE_TYPE_FOLDER = {"TVM": "TVM", "GATE": "Gates", "VALIDATOR": "Validators"}

def _mem_mb():
    # ru_maxrss is KB on Linux -> MB. This is peak RSS of the kernel process
    # so far, not just this call's delta, but printed after every family it
    # gives a clear rising trend to watch for if a future run risks the OOM
    # kernel-death pattern flagged 21-Jul-2026.
    return round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024, 1)

def _write_local_device_split(out, family, grain, table_name):
    """Best-effort local CSV split by device type, under
    PS2_pavan/<TVM|Gates|Validators|Cross_Device|Other_Device_Types>/<grain>/.
    Never allowed to raise past this function -- it is a convenience export
    for PK; the S3/manifest write above it remains the system of record."""
    try:
        grain_dir_name = f"{grain}_grain" if grain in ("device", "serial") else grain
        if DEVICE_CATEGORY_COL and DEVICE_CATEGORY_COL in out.columns:
            keyed = out[DEVICE_CATEGORY_COL].astype(str).str.upper()
            groups = list(out.groupby(keyed))
        else:
            groups = [("ALL", out)]
        for raw_key, sub in groups:
            if raw_key == "ALL":
                folder = "Cross_Device"
            else:
                folder = DEVICE_TYPE_FOLDER.get(raw_key, "Other_Device_Types")
            local_dir = os.path.join(OUT_DIR, folder, grain_dir_name)
            os.makedirs(local_dir, exist_ok=True)
            csv_path = os.path.join(local_dir, f"{table_name}.csv")
            sub.to_csv(csv_path, index=False)
        if DEVICE_CATEGORY_COL and DEVICE_CATEGORY_COL in out.columns:
            unknown = sorted(set(keyed) - set(DEVICE_TYPE_FOLDER))
            if unknown:
                print(f"  [write_output]   note: {family}/{grain} has device-category values "
                      f"not in TVM/GATE/VALIDATOR: {unknown} -> filed under Other_Device_Types/")
    except Exception as e:
        print(f"  [write_output]   WARNING local device-type split failed for {family}/{grain} "
              f"({type(e).__name__}: {str(e)[:160]}) -- S3/RDS write above is unaffected.")

def write_output(frame, family, grain):
    """
    frame  : pandas DataFrame, the computed metric table (NOT yet carrying the
             mandatory columns -- this function adds them).
    family : short family name, becomes part of the RDS table name:
             ps2_<family>_<grain>   e.g. family="phi_matrix", grain="serial"
             -> ps2_phi_matrix_serial
    grain  : one of "device" | "serial" | "subsystem" | "facility"
    Writes s3://EXPORT_BUCKET/EXPORT_PREFIX/<table_name>/computed_date=<CDATE>/run_id=<RUN_ID>/part-0.parquet
    then a manifest.json as the LAST object (the S3 trigger fires on manifest.json only).
    ALSO writes a local CSV copy under PS2_pavan/<device type>/<grain>/<table>.csv,
    split by DEVICE_CATEGORY_COL when that column is present on `frame`.
    """
    if frame is None or len(frame) == 0:
        print(f"  [write_output] {family:<24} grain={grain:<10} SKIPPED (0 rows)")
        return None

    table_name = f"ps2_{family}_{grain}" if grain != "device" or f"ps2_{family}" not in (
        "ps2_phi_matrix", "ps2_markov_transitions", "ps2_conditional_prob"
    ) else f"ps2_{family}"  # keep existing device-grain table names stable where they already exist

    out = frame.copy()
    out["grain"] = grain
    out["run_id"] = RUN_ID
    out["computed_date"] = CDATE
    out["notebook_version"] = NOTEBOOK_VERSION
    out["city_id"] = CITY_ID

    # Postgres identifier safety: lowercase, snake_case, <=63 chars -- cheap to
    # check here, expensive to discover as a Lambda CREATE TABLE failure.
    bad_cols = [c for c in out.columns if len(c) > 63 or c != c.lower()]
    if bad_cols:
        print(f"  [write_output] WARNING {family}/{grain}: column-name issues {bad_cols} -- "
              f"lower-casing and truncating before write.")
        out.columns = [c.lower()[:63] for c in out.columns]

    key_prefix = f"{EXPORT_PREFIX}/{table_name}/computed_date={CDATE}/run_id={RUN_ID}"
    local_path = os.path.join(OUT_DIR, "s3_export", table_name)
    os.makedirs(local_path, exist_ok=True)
    part_path = os.path.join(local_path, "part-0.parquet")
    out.to_parquet(part_path, index=False)

    s3_part_key = f"{key_prefix}/part-0.parquet"
    S3_CLIENT.upload_file(part_path, EXPORT_BUCKET, s3_part_key)

    manifest = {
        "table_name": table_name, "family": family, "grain": grain,
        "row_count": int(len(out)), "columns": list(out.columns),
        "computed_date": CDATE, "run_id": RUN_ID, "notebook_version": NOTEBOOK_VERSION,
        "s3_data_key": s3_part_key,
    }
    manifest_path = os.path.join(local_path, "manifest.json")
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)
    s3_manifest_key = f"{key_prefix}/manifest.json"
    S3_CLIENT.upload_file(manifest_path, EXPORT_BUCKET, s3_manifest_key)
    _ALL_MANIFESTS.append(manifest)

    _write_local_device_split(out, family, grain, table_name)

    print(f"  [write_output] {table_name:<32} grain={grain:<10} {len(out):>6} rows -> "
          f"s3://{EXPORT_BUCKET}/{s3_manifest_key}")

    # Memory hygiene: this notebook computes several O(n^2)-ish matrices (phi,
    # HMM fits, association rules) across potentially large serial-grain
    # frames across 15+ families. Freeing each family's working frame right
    # after it is written -- rather than letting every family's intermediate
    # DataFrame pile up for the whole run -- is a direct mitigation for the
    # kernel-death/OOM risk flagged 21-Jul-2026. mem_after is printed so a
    # future run shows the trend and a runaway family can be caught before
    # the kernel dies rather than after.
    del out
    gc.collect()
    print(f"  [write_output]   peak RSS so far: {_mem_mb():,} MB")

    return manifest

print("write_output() ready -- this is the ONLY export path to S3/RDS for this notebook's "
      f"tables. Local human-readable copies also land under {OUT_DIR}/<TVM|Gates|Validators|"
      "Cross_Device>/<device_grain|serial_grain|...>/<table>.csv")

---
# Family 1 — Phi (Matthews) Correlation
**Device grain** (reskinned from `PS2_enhanced.ipynb`): subsystem x subsystem phi matrix over the
whole device/city-grain chain table — unchanged methodology, same headline numbers as the reference
run (e.g. CSC_READER<->SCRST +183.1 scaled units) should reproduce here since the source table is
identical.

**Serial grain (NEW):** a device-grain phi of two subsystems blends every physical unit ever
installed in that slot. Re-running per serial answers "is THIS card reader unit's failure correlated
with THIS screen unit's failure" — the actual component-pairing question maintenance needs, and the
bridge into a hardware-batch/lot-defect signal. Because a full serial-by-serial matrix would be
thousands of near-empty 2x2s, we compute it as: for each subsystem pair, phi computed within each
serial's own cascade history, then report the **distribution** of per-serial phi (not just one
number) — serials whose own phi is a strong outlier vs. the device-grain phi for the same pair are
the maintenance-actionable signal.

In [ ]:
# ── Phi correlation — DEVICE grain ────────────────────────────────────────────
phi_device_matrix, PHI_SUBSYSTEMS = phi_matrix_for_grain(df_device)
print(f"Device-grain phi matrix: {phi_device_matrix.shape}, subsystems={PHI_SUBSYSTEMS}")

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(phi_device_matrix.astype(float), cmap=DIV_CMAP, center=0, annot=False,
            cbar_kws={"label": "phi"}, ax=ax)
ax.set_title("PS2 - Subsystem Phi Correlation (device/city grain)")
save_and_log(fig, os.path.join(OUT_DIR, "phi_device.png"))

phi_device_long = phi_device_matrix.reset_index().melt(id_vars="index", var_name="sub_b", value_name="phi")
phi_device_long = phi_device_long.rename(columns={"index": "sub_a"})
write_output(phi_device_long[["sub_a", "sub_b", "phi"]], "phi_matrix", "device")

In [ ]:
# ── Phi correlation — SERIAL grain (NEW) ──────────────────────────────────────
# NEW (21-Jul-2026): sourced directly from df_device grouped by DEVICE_SERIAL_NUMBER --
# there is no separate component-grain table (see the GOLD_TABLES note in Configuration).
phi_serial_records = []
if SERIAL_COL in df_device.columns and "subsystem_chain" in df_device.columns:
    n_serials = df_device[SERIAL_COL].nunique()
    print(f"Computing per-serial phi distributions across {n_serials:,} serials "
          f"(this may take a moment for a large fleet)...")
    per_serial_phi = defaultdict(list)   # (sub_a, sub_b) -> list of per-serial phi values
    for serial_id, g in df_device.groupby(SERIAL_COL):
        if len(g) < 10:            # too few cascade-days for this serial to compute phi meaningfully
            continue
        mat, subs = phi_matrix_for_grain(g)
        if mat.empty:
            continue
        for a in subs:
            for b in subs:
                if a < b:
                    per_serial_phi[(a, b)].append(mat.loc[a, b])
                    phi_serial_records.append({
                        "serial_id": str(serial_id), "sub_a": a, "sub_b": b, "phi": round(float(mat.loc[a, b]), 4),
                    })
    print(f"Computed per-serial phi for {len(phi_serial_records):,} (serial, subsystem-pair) observations "
          f"across {len(per_serial_phi):,} distinct subsystem pairs.")

    # Outlier serials: per-pair, flag serials whose phi is >2 std from the pair's own serial-level mean
    outlier_rows = []
    for (a, b), vals in per_serial_phi.items():
        if len(vals) < 5:
            continue
        arr = np.array(vals); mu, sd = arr.mean(), arr.std()
        if sd == 0:
            continue
        for rec in phi_serial_records:
            if rec["sub_a"] == a and rec["sub_b"] == b and abs(rec["phi"] - mu) > 2 * sd:
                outlier_rows.append({**rec, "pair_mean_phi": round(float(mu), 4), "pair_std_phi": round(float(sd), 4),
                                      "z_score": round(float((rec["phi"] - mu) / sd), 2)})
    print(f"  {len(outlier_rows):,} serial-pair outlier observations (>2 std from the pair's serial-level mean) "
          f"— these are the hardware-batch/lot-defect candidates.")
    df_phi_serial = pd.DataFrame(phi_serial_records)
    df_phi_serial_outliers = pd.DataFrame(outlier_rows)
    write_output(df_phi_serial, "phi_matrix", "serial")
    if len(df_phi_serial_outliers):
        write_output(df_phi_serial_outliers, "phi_outliers", "serial")
else:
    print("[phi/serial] SKIPPED — DEVICE_SERIAL_NUMBER/subsystem_chain not found on device_ps2_chains; "
          "see the schema-introspection prints in the load-Gold cell above.")

---
# Family 2 — Markov Transition Chains
**Device grain**: reused device-grain subsystem-to-subsystem transition matrix (top paths should
reproduce the reference run's SYSTEM->COMMS->SYSTEM 3-step chain, 1,719,066 occurrences).

**Serial grain (NEW, revised 21-Jul-2026)**: `gold.device_ps2_markov_matrix` was assumed to exist
from a "Phase-4" build but was confirmed **not to exist** anywhere in the catalog (Databricks
discovery notebook, exact + fuzzy search). Per-serial transition matrices are therefore always
computed directly from `device_ps2_chains`, grouped by `DEVICE_SERIAL_NUMBER` — specifically a
lightweight self-transition / short-cycle-back rate per serial, which reveals whether a *specific
physical unit* has an elevated repeat-fault rate vs. the device-level average, a leading indicator
of a failing part vs. a failing installation/environment.

In [ ]:
# ── Markov transitions — DEVICE grain ─────────────────────────────────────────
device_chains = [parse_chain(s) for s in df_device["subsystem_chain"].dropna()]
markov_states_device, markov_P_device = markov_matrix_for_chains(device_chains)
print(f"Device-grain Markov: {len(markov_states_device)} states, {len(device_chains):,} chains")

mk_recs = [{"from_sub": markov_states_device[i], "to_sub": markov_states_device[j],
            "prob": round(float(markov_P_device[i, j]), 4)}
           for i in range(len(markov_states_device)) for j in range(len(markov_states_device))
           if markov_P_device[i, j] > 0.02]
df_markov_device = pd.DataFrame(mk_recs)
write_output(df_markov_device, "markov_transitions", "device")

if len(markov_states_device):
    fig, ax = plt.subplots(figsize=(8, 7))
    sns.heatmap(pd.DataFrame(markov_P_device, index=markov_states_device, columns=markov_states_device),
                cmap=SEQ_CMAP, ax=ax)
    ax.set_title("PS2 - Markov Transition Matrix (device/city grain)")
    save_and_log(fig, os.path.join(OUT_DIR, "markov_device.png"))

In [ ]:
# ── Markov transitions — SERIAL grain (NEW) ───────────────────────────────────
# NEW (21-Jul-2026): gold.device_ps2_markov_matrix never existed (confirmed via the
# Databricks discovery notebook) -- this always computes the lightweight per-serial
# self-transition rate directly from device_ps2_chains, grouped by DEVICE_SERIAL_NUMBER.
if SERIAL_COL in df_device.columns:
    recs = []
    for serial_id, g in df_device.groupby(SERIAL_COL):
        chains = [parse_chain(s) for s in g.get("subsystem_chain", pd.Series(dtype=object)).dropna()]
        chains = [c for c in chains if len(c) >= 2]
        if len(chains) < 5:
            continue
        self_trans = sum(1 for c in chains for a, b in zip(c[:-1], c[1:]) if a == b)
        total_trans = sum(len(c) - 1 for c in chains)
        recs.append({"serial_id": str(serial_id), "n_chains": len(chains),
                      "self_transition_rate": round(self_trans / max(total_trans, 1), 4)})
    df_markov_serial_fallback = pd.DataFrame(recs).sort_values("self_transition_rate", ascending=False)
    write_output(df_markov_serial_fallback, "markov_self_transition", "serial")
    if len(df_markov_serial_fallback):
        print(f"Serial-grain self-transition rate computed for {len(df_markov_serial_fallback):,} serials "
              "(>=5 multi-hop chains each) -- top values are chronically re-faulting physical units.")
        print(df_markov_serial_fallback.head(10).to_string(index=False))
else:
    print("[markov/serial] SKIPPED — no serial-grain source available.")

---
# Family 3 — HMM 3-state Regime (Critical / Moderate / Minor)
**Device grain**: reused 3-state Gaussian HMM on per-device cascade severity (dwell times should
reproduce the reference run: Critical 2.6% / 1.5d dwell, Minor 21% / 2.0d, Moderate 76.4% / 6.3d).

**Serial grain (NEW)**: dwell-time-in-Critical is inherently a component-lifecycle question — a
device-grain HMM conflates "five different card readers, four fine" with "one bad reader." Fit a
lighter single-feature HMM per serial (chain_length as the emission) where support allows.

In [ ]:
# ── HMM 3-state — DEVICE grain ────────────────────────────────────────────────
device_hmm_result = fit_hmm_3state(df_device["chain_length"].fillna(0).values)
if device_hmm_result:
    model, order = device_hmm_result["model"], device_hmm_result["order"]
    mu, sigma = device_hmm_result["mu"], device_hmm_result["sigma"]
    labels = ["minor", "moderate", "critical"]
    means = (model.means_.flatten()[order] * sigma) + mu   # back to original chain_length units
    print("Device-grain HMM state means (chain_length):", dict(zip(labels, np.round(means, 2))))
    if not device_hmm_result["converged"]:
        print(f"[hmm/device] NOTE: EM did not fully converge within "
              f"{device_hmm_result['n_iter_run']} iterations (recorded in the output "
              f"table's 'converged' column below, not hidden) -- the state split above is "
              f"still well-separated and usable, but treat it as slightly less certain "
              f"than a cleanly-converged fit.")
    X_std = (df_device["chain_length"].fillna(0).values.reshape(-1, 1) - mu) / sigma
    states_seq = model.predict(X_std)
    remapped = np.zeros_like(states_seq)
    for new_i, old_i in enumerate(order):
        remapped[states_seq == old_i] = new_i
    df_device["_hmm_state"] = [labels[i] for i in remapped]
    hmm_summary = df_device["_hmm_state"].value_counts(normalize=True).mul(100).round(2)
    print("State occupancy %:\n", hmm_summary)
    df_hmm_device = pd.DataFrame({"state": hmm_summary.index, "pct_time": hmm_summary.values})
    df_hmm_device["converged"] = device_hmm_result["converged"]
    df_hmm_device["n_iter_run"] = device_hmm_result["n_iter_run"]
    write_output(df_hmm_device, "hmm_regimes", "device")
else:
    print("[hmm/device] insufficient rows to fit")

In [ ]:
# ── HMM 3-state — SERIAL grain (NEW) ──────────────────────────────────────────
# NEW (21-Jul-2026): sourced from df_device grouped by DEVICE_SERIAL_NUMBER.
hmm_serial_records = []
if SERIAL_COL in df_device.columns and "chain_length" in df_device.columns:
    fit_count, skip_count, nonconverged_count = 0, 0, 0
    for serial_id, g in df_device.groupby(SERIAL_COL):
        vals = g["chain_length"].fillna(0).values
        if len(vals) < 30:
            skip_count += 1
            continue
        result = fit_hmm_3state(vals)
        if not result:
            skip_count += 1
            continue
        model, order = result["model"], result["order"]
        mu, sigma = result["mu"], result["sigma"]
        labels = ["minor", "moderate", "critical"]
        X_std = (vals.reshape(-1, 1) - mu) / sigma
        states_seq = model.predict(X_std)
        remapped = np.array([labels[list(order).index(s)] for s in states_seq])
        crit_pct = float((remapped == "critical").mean() * 100)
        if not result["converged"]:
            nonconverged_count += 1
        hmm_serial_records.append({"serial_id": str(serial_id), "n_obs": int(len(vals)),
                                    "pct_time_critical": round(crit_pct, 2),
                                    "mean_chain_length": round(float(vals.mean()), 2),
                                    "converged": result["converged"],
                                    "n_iter_run": result["n_iter_run"]})
        fit_count += 1
    print(f"HMM fit for {fit_count:,} serials (>=30 obs each); {skip_count:,} serials skipped for insufficient data "
          f"(reported, not silently dropped); {nonconverged_count:,}/{fit_count:,} fits did not fully converge "
          f"within the iteration budget (flagged in the 'converged' column, not hidden -- state splits are "
          f"typically still usable, see the device-grain cell's note above).")
    df_hmm_serial = pd.DataFrame(hmm_serial_records).sort_values("pct_time_critical", ascending=False)
    write_output(df_hmm_serial, "hmm_regimes", "serial")
    if len(df_hmm_serial):
        print("\nTop 15 serials by % time in Critical regime (chronic-unit candidates):")
        print(df_hmm_serial.head(15).to_string(index=False))
else:
    print("[hmm/serial] SKIPPED — DEVICE_SERIAL_NUMBER/chain_length not found on device_ps2_chains.")

---
# Family 4 — Conditional Probability P(B | A, T)
**Device grain**: reused from `PS2_enhanced.ipynb` per time-window (reference numbers e.g.
P(ALARM->SYSTEM)=0.998 in the 5-15min window should reproduce).

**Serial grain (NEW, support-floored):** the same conditional probability computed per serial will
have far smaller per-cell counts once split five ways by window and further by serial. Cells below
`MIN_SUPPORT_SERIAL_COND_PROB` observations are **suppressed and counted**, never reported as a
0.998-style number off of n=3 — see the suppression summary printed at the end of this section.

In [ ]:
# ── Conditional probability P(B|A,T) — DEVICE grain ───────────────────────────
def cond_prob_table(df_sub, chain_col="subsystem_chain", span_col="chain_span_min", min_n=0):
    subs = get_all_subsystems(df_sub, chain_col)
    recs, suppressed = [], 0
    for wname, (lo, hi) in TIME_WINDOWS.items():
        win = df_sub[(df_sub[span_col].fillna(0) >= lo) & (df_sub[span_col].fillna(0) < hi)] if span_col in df_sub else df_sub
        if len(win) == 0:
            continue
        flags = chain_to_binary_flags(win, subs, chain_col).astype(bool)
        for a in subs:
            n_a = int(flags[a].sum())
            if n_a == 0:
                continue
            for b in subs:
                if a == b:
                    continue
                n_ab = int((flags[a] & flags[b]).sum())
                if suppressed_floor(n_a, min_n):
                    suppressed += 1
                    continue
                recs.append({"sub_a": a, "sub_b": b, "window": wname, "n_a": n_a, "n_ab": n_ab,
                             "p_b_given_a": round(n_ab / n_a, 4)})
    return pd.DataFrame(recs), suppressed

df_cond_device, supp_device = cond_prob_table(df_device, min_n=0)
print(f"Device-grain conditional-probability rows: {len(df_cond_device):,} (no support floor at device grain — "
      f"counts are large enough)")
write_output(df_cond_device, "conditional_prob", "device")

In [ ]:
# ── Conditional probability P(B|A,T) — SERIAL grain (NEW, support-floored) ────
# NEW (21-Jul-2026): sourced from df_device grouped by DEVICE_SERIAL_NUMBER.
if SERIAL_COL in df_device.columns and "subsystem_chain" in df_device.columns:
    span_col = "chain_span_min" if "chain_span_min" in df_device.columns else None
    all_recs, total_suppressed = [], 0
    for serial_id, g in df_device.groupby(SERIAL_COL):
        if len(g) < 10:
            total_suppressed += 1
            continue
        tbl, supp = cond_prob_table(g, span_col=span_col or "subsystem_chain",
                                     min_n=MIN_SUPPORT_SERIAL_COND_PROB)
        total_suppressed += supp
        if len(tbl):
            tbl.insert(0, "serial_id", str(serial_id))
            all_recs.append(tbl)
    df_cond_serial = pd.concat(all_recs, ignore_index=True) if all_recs else pd.DataFrame()
    print(f"Serial-grain conditional-probability: {len(df_cond_serial):,} rows survived the "
          f"min-support floor (n>={MIN_SUPPORT_SERIAL_COND_PROB}); {total_suppressed:,} "
          f"(serial, window, pair) cells SUPPRESSED for insufficient support (reported, not dropped).")
    write_output(df_cond_serial, "conditional_prob", "serial")
    suppression_summary = pd.DataFrame([{"family": "conditional_prob", "grain": "serial",
                                          "min_support_floor": MIN_SUPPORT_SERIAL_COND_PROB,
                                          "cells_suppressed": total_suppressed,
                                          "cells_reported": len(df_cond_serial)}])
    write_output(suppression_summary, "suppression_summary", "serial")
else:
    print("[cond_prob/serial] SKIPPED — no serial-grain source available.")

---
# Family 5 — FP-growth Association Rules
**Device grain**: reused (reference top lift 9.678, 36 rules all lift>5).

**Serial grain (NEW, support-floored):** `MIN_SUPPORT_SERIAL_ASSOC` (0.02) itemset-support floor
applied explicitly via `apriori(..., min_support=MIN_SUPPORT_SERIAL_ASSOC)`; rules below the floor
never enter the candidate set in the first place (mlxtend enforces this at generation time, so there
is no separate suppression count to report here — the floor IS the mechanism).

In [ ]:
# ── Association rules — DEVICE grain ──────────────────────────────────────────
def assoc_rules_for(df_sub, chain_col="subsystem_chain", min_support=0.01):
    baskets = [parse_chain(s) for s in df_sub[chain_col].dropna()]
    baskets = [b for b in baskets if len(b) >= 2]
    if len(baskets) < 20:
        return pd.DataFrame()
    te = TransactionEncoder()
    te_ary = te.fit(baskets).transform(baskets)
    onehot = pd.DataFrame(te_ary, columns=te.columns_)
    freq = apriori(onehot, min_support=min_support, use_colnames=True)
    if freq.empty:
        return pd.DataFrame()
    rules = association_rules(freq, metric="lift", min_threshold=1.0)
    rules["antecedents"] = rules["antecedents"].apply(lambda s: ",".join(sorted(s)))
    rules["consequents"] = rules["consequents"].apply(lambda s: ",".join(sorted(s)))
    return rules[["antecedents", "consequents", "support", "confidence", "lift", "conviction"]].round(4)

df_assoc_device = assoc_rules_for(df_device, min_support=0.01)
print(f"Device-grain association rules: {len(df_assoc_device):,} (min_support=0.01)")
write_output(df_assoc_device, "association_rules", "device")

In [ ]:
# ── Association rules — SERIAL grain (NEW, support-floored) ──────────────────
# NEW (21-Jul-2026): sourced from df_device grouped by DEVICE_SERIAL_NUMBER; still
# capped at the top-200 richest serials by volume (unchanged cost guardrail).
if SERIAL_COL in df_device.columns:
    top_serials = df_device[SERIAL_COL].value_counts().head(200).index  # richest 200 serials by cascade-day volume
    all_rules = []
    fit_count, skip_count = 0, 0
    for serial_id in top_serials:
        g = df_device[df_device[SERIAL_COL] == serial_id]
        rules = assoc_rules_for(g, min_support=MIN_SUPPORT_SERIAL_ASSOC)
        if len(rules):
            rules.insert(0, "serial_id", str(serial_id))
            all_rules.append(rules)
            fit_count += 1
        else:
            skip_count += 1
    df_assoc_serial = pd.concat(all_rules, ignore_index=True) if all_rules else pd.DataFrame()
    print(f"Serial-grain association rules: {len(df_assoc_serial):,} rules across {fit_count:,} serials "
          f"(of top-200 by volume); {skip_count:,} serials produced no rule >= "
          f"{MIN_SUPPORT_SERIAL_ASSOC} support.")
    write_output(df_assoc_serial, "association_rules", "serial")
else:
    print("[assoc/serial] SKIPPED — no serial-grain source available.")

---
# Family 6 — Network Centrality (subsystem-cascade graph)
**Device-category grain ONLY — deliberately NOT re-run per serial.** The graph nodes are subsystem
TYPES (SYSTEM, COMMS, PRINTER, ...), not device or serial instances; centrality is a property of the
subsystem taxonomy. A serial-level graph would just be thousands of disconnected tiny components with
no topology — see the grain-policy note at the top of this notebook. This section reproduces the
reference run (PRINTER betweenness 0.819, SYSTEM pagerank 0.241) and additionally splits by device
category (TVM/GATE/VALIDATOR), which the reference set only partially did.

In [ ]:
# ── Network centrality — subsystem graph, all + per device-category ──────────
def build_cascade_graph(df_sub, chain_col="subsystem_chain"):
    G = nx.DiGraph()
    for chain in df_sub[chain_col].dropna():
        seq = parse_chain(chain)
        for a, b in zip(seq[:-1], seq[1:]):
            if G.has_edge(a, b):
                G[a][b]["weight"] += 1
            else:
                G.add_edge(a, b, weight=1)
    return G

def centrality_table(G, scope):
    if G.number_of_nodes() == 0:
        return pd.DataFrame()
    bc = nx.betweenness_centrality(G, weight="weight")
    pr = nx.pagerank(G, weight="weight")
    recs = [{"scope": scope, "subsystem": n, "betweenness": round(bc.get(n, 0), 4),
             "pagerank": round(pr.get(n, 0), 4), "in_degree": G.in_degree(n, weight="weight"),
             "out_degree": G.out_degree(n, weight="weight")} for n in G.nodes()]
    return pd.DataFrame(recs).sort_values("betweenness", ascending=False)

cent_all = centrality_table(build_cascade_graph(df_device), "ALL")
print("Top routers (betweenness), all devices:")
print(cent_all.head(5).to_string(index=False))

cent_frames = [cent_all]
if "mars_device_category" in df_device.columns:
    for cat in DEVICE_CATEGORIES:
        sub = df_device[df_device["mars_device_category"] == cat]
        if len(sub) >= 20:
            cent_frames.append(centrality_table(build_cascade_graph(sub), cat))

df_centrality = pd.concat(cent_frames, ignore_index=True)
write_output(df_centrality, "network_centrality", "subsystem")

---
# Family 7 — Cascade Velocity / Speed Windows
**Device grain, stratified by serial-age-at-event (NEW addition to this family).** Per the grain
policy, a full per-serial velocity distribution is too sparse for most units to be a distribution at
all — instead we bucket each cascade-day by how old the responsible serial was at the time (using
`REPORTED_CHANGED_DTM` as the install-proxy, matching the 17-Jul data-foundation convention) and
report velocity by age-bucket. This answers "do newly installed units cascade faster/slower than
aged units" without the sparsity problem of a true per-serial rerun.

In [ ]:
# ── Cascade velocity — device grain + serial-age stratification (NEW) ────────
df_device["window_bucket"] = pd.cut(
    df_device["chain_span_min"].fillna(0),
    bins=[-1, 5, 15, 30, 60, 1e9],
    labels=list(TIME_WINDOWS.keys()),
)
vel_recs = []
for wname in TIME_WINDOWS:
    sub = df_device[df_device["window_bucket"] == wname]
    if len(sub) == 0:
        continue
    vel = (sub["chain_length"] / sub["chain_span_min"].replace(0, np.nan)).mean()
    vel_recs.append({"window": wname, "n": int(len(sub)),
                      "mean_chain_length": round(float(sub["chain_length"].mean()), 3),
                      "mean_velocity_min_per_fault": round(float(vel), 3) if vel == vel else None})
df_velocity_device = pd.DataFrame(vel_recs)
write_output(df_velocity_device, "cascade_velocity", "device")

# NEW (21-Jul-2026): age-stratified velocity, using the SERIAL_AGE_DAYS lookup built
# in the loader cell. IMPORTANT SCOPE NOTE: SERIAL_AGE_DAYS is each serial's age AS OF
# TODAY (real, from hw_config_current, or a cascade-history-span proxy -- see loader
# cell print for which), not its age at the time of each individual historical event.
# So this answers "do currently-older physical units show different aggregate cascade
# velocity than currently-newer ones" -- a real, useful question -- NOT "how does a
# given unit's velocity change as it ages" (that would need point-in-time install
# history this notebook does not have access to). Labeled clearly below, not hidden.
age_strat_recs = []
if SERIAL_COL in df_device.columns and "chain_span_min" in df_device.columns and SERIAL_AGE_DAYS:
    tmp = df_device.copy()
    tmp["serial_age_days"] = tmp[SERIAL_COL].astype(str).map(SERIAL_AGE_DAYS)
    tmp = tmp.dropna(subset=["serial_age_days"])
    tmp = tmp[tmp["serial_age_days"].between(0, 3650)]
    tmp["age_bucket"] = pd.cut(tmp["serial_age_days"],
                                bins=[0, 90, 365, 730, 1825, 99999],
                                labels=["0-90d", "90-365d", "1-2yr", "2-5yr", "5yr+"])
    for bucket, g in tmp.groupby("age_bucket", observed=True):
        if len(g) < 20:
            continue
        vel = (g["chain_length"] / g["chain_span_min"].replace(0, np.nan)).mean() if "chain_length" in g else None
        age_strat_recs.append({"age_bucket": str(bucket), "n": int(len(g)),
                                "mean_velocity_min_per_fault": round(float(vel), 3) if vel == vel and vel is not None else None})
    print(f"Cascade velocity by CURRENT serial age (source: {SERIAL_AGE_SOURCE}):")
    print(pd.DataFrame(age_strat_recs).to_string(index=False))
else:
    print("[velocity/age-stratified] SKIPPED — SERIAL_AGE_DAYS is empty or chain_span_min missing.")
df_velocity_age = pd.DataFrame(age_strat_recs)
write_output(df_velocity_age, "cascade_velocity_by_age", "serial")

---
# Family 8 — Device / Serial Recurrence (Chronic vs Sporadic)
**Both grains re-run natively.** Recurrence-of-failure is fundamentally a property of the physical
part, so the serial-grain version (renamed here to make that explicit) is what should drive
"replace this unit" decisions; the device-grain version stays because a slot that eats parts is
itself a signal (bad environment, power quality, vibration) — kept for facility/dispatch staffing.

In [ ]:
# ── Recurrence — DEVICE grain (reused) ────────────────────────────────────────
dev_counts = df_device.groupby("DEVICE_ID").size()
chronic_cut = dev_counts.quantile(0.90)
rec_recs = [{"device_id": str(d), "cascade_days": int(n),
             "chronic": bool(n >= chronic_cut)} for d, n in dev_counts.items()]
df_recurrence_device = pd.DataFrame(rec_recs).sort_values("cascade_days", ascending=False)
if DEVICE_CATEGORY_COL and DEVICE_ID_TO_CATEGORY:
    df_recurrence_device[DEVICE_CATEGORY_COL] = df_recurrence_device["device_id"].map(DEVICE_ID_TO_CATEGORY)
print(f"Device-grain: {int((df_recurrence_device['chronic']).sum())} chronic devices "
      f"(>= {chronic_cut:.0f} cascade-days, 90th pct)")
write_output(df_recurrence_device, "recurrence", "device")

In [ ]:
# ── Recurrence — SERIAL grain (NEW, primary maintenance signal) ──────────────
# NEW (21-Jul-2026): sourced from df_device grouped by DEVICE_SERIAL_NUMBER.
if SERIAL_COL in df_device.columns:
    ser_counts = df_device.groupby(SERIAL_COL).size()
    chronic_cut_serial = ser_counts.quantile(0.90)
    rec_recs_serial = [{"serial_id": str(s), "cascade_days": int(n),
                         "chronic": bool(n >= chronic_cut_serial)} for s, n in ser_counts.items()]
    df_recurrence_serial = pd.DataFrame(rec_recs_serial).sort_values("cascade_days", ascending=False)
    n_chronic = int(df_recurrence_serial["chronic"].sum())
    print(f"Serial-grain: {n_chronic:,} chronic serials (>= {chronic_cut_serial:.0f} cascade-days, 90th pct) "
          f"of {len(df_recurrence_serial):,} total serials.")
    write_output(df_recurrence_serial, "recurrence", "serial")
else:
    print("[recurrence/serial] SKIPPED — no serial-grain source available.")

---
# Family 9 — Business Impact Scoring
**Both grains.** Device-grain informs facility staffing/dispatch; serial-grain answers "which
physical units are costing the most" for spares/warranty conversations — a different procurement
question, kept as a separate table rather than collapsed into one.

In [ ]:
# ── Business impact — DEVICE + SERIAL grain ───────────────────────────────────
SEVERITY_WEIGHTS = {"INFO": 1, "WARNING": 3, "MAJOR": 7, "CRITICAL": 15}

def business_impact_for(df_sub, group_col):
    def sev_score(chain):
        segs = parse_chain(chain)
        return sum(SEVERITY_WEIGHTS.get(s.upper(), 2) for s in segs)
    tmp = df_sub.copy()
    tmp["_impact"] = tmp.get("severity_chain", pd.Series(dtype=object)).apply(sev_score) if "severity_chain" in tmp else 0
    g = tmp.groupby(group_col).agg(cascade_days=("_impact", "size"), total_impact=("_impact", "sum"),
                                    avg_impact=("_impact", "mean")).reset_index()
    g = g.rename(columns={group_col: "entity_id"})
    return g.sort_values("total_impact", ascending=False)

df_impact_device = business_impact_for(df_device, "DEVICE_ID")
if DEVICE_CATEGORY_COL and DEVICE_ID_TO_CATEGORY:
    df_impact_device[DEVICE_CATEGORY_COL] = df_impact_device["entity_id"].map(DEVICE_ID_TO_CATEGORY)
write_output(df_impact_device, "business_impact", "device")
print("Top 10 devices by business impact:\n", df_impact_device.head(10).to_string(index=False))

# NEW (21-Jul-2026): serial-grain business impact now sourced from df_device itself
# (grouped by DEVICE_SERIAL_NUMBER) instead of the nonexistent component-day table.
if SERIAL_COL in df_device.columns:
    df_impact_serial = business_impact_for(df_device, SERIAL_COL)
    write_output(df_impact_serial, "business_impact", "serial")
    print("\nTop 10 serials by business impact:\n", df_impact_serial.head(10).to_string(index=False))
else:
    print("[business_impact/serial] SKIPPED — no serial-grain source available.")

---
# Family 10 — Facility-Level Cascade Contagion
**Facility grain only — deliberately not re-run per serial**, per the grain policy: this is
inherently a facility/station aggregation question (does a fault at one device raise failure odds at
sibling devices in the same facility). Reused from `PS2_enhanced.ipynb`.

In [ ]:
# ── Facility contagion — FACILITY grain (reused, unchanged methodology) ──────
if "FACILITY_ID" in df_device.columns:
    fac = df_device.groupby("FACILITY_ID").agg(
        cascade_days=("DEVICE_ID", "size"), distinct_devices=("DEVICE_ID", "nunique")
    ).reset_index()
    fac["contagion_rate"] = (fac["cascade_days"] / fac["distinct_devices"].replace(0, np.nan)).round(3)
    df_facility = fac.sort_values("contagion_rate", ascending=False)
    write_output(df_facility, "facility_contagion", "facility")
    print("Top 10 facilities by contagion rate:\n", df_facility.head(10).to_string(index=False))
else:
    print("[facility_contagion] SKIPPED — FACILITY_ID not in df_device.")

---
# Family 11 — Cascade Ignition & Termination
**Subsystem/device-category grain only** — this is a subsystem-role question (which types tend to
start vs. end cascades), same reasoning as network centrality. Reused from `PS2_enhanced.ipynb`.

In [ ]:
# ── Ignition / termination — subsystem grain (reused) ─────────────────────────
def first_last(chain):
    seq = parse_chain(chain)
    return (seq[0], seq[-1]) if len(seq) >= 1 else (None, None)

firsts, lasts = zip(*df_device["subsystem_chain"].dropna().apply(first_last)) if len(df_device) else ([], [])
ign = pd.Series(firsts).value_counts()
term = pd.Series(lasts).value_counts()
df_ignition = pd.DataFrame({"subsystem": ign.index, "ignition_count": ign.values}).merge(
    pd.DataFrame({"subsystem": term.index, "termination_count": term.values}), on="subsystem", how="outer"
).fillna(0)
write_output(df_ignition, "ignition_termination", "subsystem")
print(df_ignition.sort_values("ignition_count", ascending=False).head(10).to_string(index=False))

---
# NEW Family 12 — Serial-Level Chronic-Cascade Recurrence Scoring

**This is explicitly a descriptive, empirical recurrence rate — NOT a fitted survival model.**
PS5 owns survival/RUL modeling (Cox/Weibull, model-based, time-to-failure). This section computes a
simple empirical hazard curve (cascades-since-install vs. time) and a chronicity score (cascades per
install-day vs. the peer-serial distribution within the same device-category/subsystem), with no
fitted model and no censoring-aware estimator. If this boundary ever needs revisiting, that is a
decision for PK/the modeling lead, not something to blur silently in a notebook cell.

In [ ]:
# ── NEW: Chronic-cascade recurrence (serial, descriptive/empirical only) ──────
# NEW (21-Jul-2026): sourced from df_device grouped by DEVICE_SERIAL_NUMBER; the
# "reference period" denominator is SERIAL_AGE_DAYS (built in the loader cell -- real
# component age from hw_config_current if available, else a cascade-history-span proxy).
# See SERIAL_AGE_SOURCE for which one applies to this run -- printed below, not hidden.
if SERIAL_COL in df_device.columns and SERIAL_AGE_DAYS:
    tmp = df_device.copy()
    tmp["_ref_period_days"] = tmp[SERIAL_COL].astype(str).map(SERIAL_AGE_DAYS)
    tmp = tmp.dropna(subset=["_ref_period_days"])
    tmp = tmp[tmp["_ref_period_days"] > 0]
    if DEVICE_CATEGORY_COL and DEVICE_ID_TO_CATEGORY and "DEVICE_ID" in tmp.columns:
        tmp["_device_category"] = tmp["DEVICE_ID"].map(DEVICE_ID_TO_CATEGORY)
    else:
        tmp["_device_category"] = None

    chronic_recs = []
    for serial_id, g in tmp.groupby(SERIAL_COL):
        ref_period_days = float(g["_ref_period_days"].iloc[0])
        n_cascades = len(g)
        recurrence_rate = n_cascades / ref_period_days   # empirical cascades/day over the reference period
        chronic_recs.append({
            "serial_id": str(serial_id),
            "device_category": g["_device_category"].iloc[0] if "_device_category" in g else None,
            "reference_period_days": int(ref_period_days),
            "reference_period_source": SERIAL_AGE_SOURCE,
            "n_cascades": int(n_cascades),
            "recurrence_rate_per_day": round(recurrence_rate, 5),
        })
    df_chronic = pd.DataFrame(chronic_recs)
    if len(df_chronic) and df_chronic["device_category"].notna().any():
        # chronicity score = recurrence_rate vs. peer distribution within the same device_category
        df_chronic["peer_pct_rank"] = df_chronic.groupby("device_category")["recurrence_rate_per_day"] \
            .rank(pct=True).round(3)
    elif len(df_chronic):
        print("[chronic_recurrence] no device_category available for peer grouping -- ranking across the whole fleet instead.")
        df_chronic["peer_pct_rank"] = df_chronic["recurrence_rate_per_day"].rank(pct=True).round(3)
    else:
        df_chronic["peer_pct_rank"] = pd.Series(dtype=float)
    df_chronic["chronicity_flag"] = df_chronic["peer_pct_rank"] >= 0.90 if len(df_chronic) else pd.Series(dtype=bool)
    print(f"Chronic-cascade recurrence: {len(df_chronic):,} serials scored (reference period source: "
          f"{SERIAL_AGE_SOURCE}); {int(df_chronic['chronicity_flag'].sum()) if len(df_chronic) else 0:,} "
          f"flagged >=90th percentile recurrence rate within their peer group.")
    write_output(df_chronic, "chronic_recurrence", "serial")
else:
    print("[chronic_recurrence] SKIPPED — SERIAL_AGE_DAYS is empty or no serial-grain source available.")

---
# NEW Family 13 — Cross-PS Cascade Attribution

Joins `device_cross_ps` / `serial_cross_ps` (the 17-Jul cross-PS join spine) to compute simple
**correlational co-occurrence rates**: of the devices/serials that ignite PS2 cascades, what fraction
also carry a PS1 elevated-risk flag, a PS3 severity/root-cause incident, or a PS4 anomaly flag in the
same period. This is the single most differentiated addition relative to the reference 31-chart set,
and is exactly the data the dashboard's cross-tab linkage (instruction 6) needs.

**Guardrail: this section only READS PS1/PS3/PS4's existing flags — it never recomputes anomaly
detection, severity classification, or failure probability.** Language stays strictly correlational
("N% co-occur with"), never causal.

In [ ]:
# ── NEW: Cross-PS cascade attribution (correlational, read-only on PS1/3/4) ──
# NEW (21-Jul-2026): gold.device_cross_ps / serial_cross_ps never existed (confirmed via
# the Databricks discovery notebook). Instead, this builds the device-level cross-PS flag
# spine INLINE from the 3 real gold tables (device_ps1_daily, device_ps3_incident,
# device_ps4_hourly), each defensively column-resolved and reported, then reuses the same
# cross_ps_attribution() co-occurrence logic as before (unchanged below).
def cross_ps_attribution(df_cross, id_col, ignition_ids, grain_label):
    if df_cross is None or id_col not in df_cross.columns:
        print(f"[cross_ps_attribution/{grain_label}] SKIPPED — cross-PS table or id column not available.")
        return pd.DataFrame()
    flag_cols = {
        "ps1_high_risk": next((c for c in df_cross.columns if "ps1" in c.lower() and
                                ("risk" in c.lower() or "fail" in c.lower())), None),
        "ps3_severity_incident": next((c for c in df_cross.columns if "ps3" in c.lower() and
                                        ("sever" in c.lower() or "incident" in c.lower())), None),
        "ps4_anomaly": next((c for c in df_cross.columns if "ps4" in c.lower() and "anomal" in c.lower()), None),
    }
    resolved = {k: v for k, v in flag_cols.items() if v is not None}
    print(f"[cross_ps_attribution/{grain_label}] resolved flag columns: {resolved}")
    if not resolved:
        print(f"[cross_ps_attribution/{grain_label}] no PS1/3/4 flag-like columns found on {df_cross.columns.tolist()} "
              f"— recording ignition-id overlap only, no co-occurrence rate.")

    ignition_set = set(str(x) for x in ignition_ids)
    sub = df_cross[df_cross[id_col].astype(str).isin(ignition_set)].copy()
    n_total = len(ignition_set)
    n_matched = len(sub)
    print(f"  {n_matched:,} / {n_total:,} PS2-ignition {grain_label}s found in the cross-PS spine "
          f"({n_matched/max(n_total,1):.1%} match rate).")

    summary_recs = [{"entity_grain": grain_label, "n_ps2_ignition_entities": n_total,
                      "n_matched_in_cross_ps": n_matched,
                      "match_rate": round(n_matched / max(n_total, 1), 4)}]
    for flag_name, col in resolved.items():
        co_occur = int(sub[col].fillna(False).astype(bool).sum()) if col in sub else 0
        rate = round(co_occur / max(n_matched, 1), 4)
        summary_recs[0][f"{flag_name}_co_occur_n"] = co_occur
        summary_recs[0][f"{flag_name}_co_occur_rate"] = rate
        print(f"  {flag_name}: {co_occur:,}/{n_matched:,} ({rate:.1%}) of PS2-ignition {grain_label}s "
              f"co-occur with this flag (CORRELATIONAL — not causal, and PS4's anomaly logic is read, not re-derived).")
    return pd.DataFrame(summary_recs)


def _coerce_flag(s):
    """Best-effort boolean coercion for a column that isn't guaranteed to already be boolean."""
    if s.dtype == bool:
        return s.fillna(False)
    if pd.api.types.is_numeric_dtype(s):
        return s.fillna(0) > 0
    return s.astype(str).str.upper().isin(["Y", "YES", "TRUE", "1", "HIGH", "CRITICAL", "SEVERE"])


def _build_flag(schema_and_uri, id_col, keyword_subs, flag_name):
    """NEW (21-Jul-2026, OOM fix): takes (columns, s3_uri) from a peek_gold_schema() call
    (cell 7) instead of an already-fully-loaded frame. Resolves the flag-like column by
    keyword substring match against the SCHEMA ONLY, then reads ONLY [id_col, matched_col]
    from S3 -- not the whole table -- which keeps memory bounded even for a large
    hourly-grain source like device_ps4_hourly."""
    cols, s3_uri = schema_and_uri if schema_and_uri else (None, None)
    if not cols or id_col not in cols:
        print(f"[cross_ps_build] {flag_name}: source table unavailable or missing {id_col} — skipped.")
        return None
    col = next((c for c in cols if any(k in c.lower() for k in keyword_subs)), None)
    if col is None:
        print(f"[cross_ps_build] {flag_name}: no matching column found in {cols} — skipped.")
        return None
    try:
        tmp = pd.read_parquet(s3_uri, columns=[id_col, col],
                               storage_options={"client_kwargs": {"region_name": REGION}})
    except Exception as e:
        print(f"[cross_ps_build] {flag_name}: column-limited read of '{col}' FAILED "
              f"({type(e).__name__}: {str(e)[:160]}) — skipped.")
        return None
    tmp[flag_name] = _coerce_flag(tmp[col])
    out = tmp.groupby(id_col)[flag_name].max().reset_index()
    del tmp
    print(f"[cross_ps_build] {flag_name}: resolved from column '{col}' (column-limited read -- "
          f"only {id_col}+{col} pulled from S3, not the whole table), "
          f"{int(out[flag_name].sum()):,}/{len(out):,} devices flagged.")
    return out


_flag_frames = []
for _f in (
    _build_flag((PS1_SCHEMA_COLS, PS1_URI), "DEVICE_ID", ["risk", "fail", "predict"], "ps1_high_risk"),
    _build_flag((PS3_SCHEMA_COLS, PS3_URI), "DEVICE_ID", ["sever", "incident"], "ps3_severity_incident"),
    _build_flag((PS4_SCHEMA_COLS, PS4_URI), "DEVICE_ID", ["anomal"], "ps4_anomaly"),
):
    if _f is not None:
        _flag_frames.append(_f)

if _flag_frames:
    df_crossps_device_built = _flag_frames[0]
    for _f in _flag_frames[1:]:
        df_crossps_device_built = df_crossps_device_built.merge(_f, on="DEVICE_ID", how="outer")
else:
    df_crossps_device_built = pd.DataFrame(columns=["DEVICE_ID"])

device_ignition_ids = df_device["DEVICE_ID"].unique().tolist() if "DEVICE_ID" in df_device.columns else []
df_crossps_device = cross_ps_attribution(df_crossps_device_built, "DEVICE_ID", device_ignition_ids, "device")
write_output(df_crossps_device, "cross_ps_attribution", "device")

# Serial grain: PS1/PS3/PS4 are DEVICE-keyed, not serial-keyed -- inherit the device-level
# flags onto each serial via its parent DEVICE_ID. Explicitly labeled below as inherited,
# not independently serial-resolved (there is no serial-keyed PS1/PS3/PS4 source to join to).
if SERIAL_COL in df_device.columns and "DEVICE_ID" in df_device.columns and _flag_frames:
    serial_device_map = df_device[[SERIAL_COL, "DEVICE_ID"]].drop_duplicates()
    df_crossps_serial_built = serial_device_map.merge(df_crossps_device_built, on="DEVICE_ID", how="left")
    serial_ignition_ids = df_device[SERIAL_COL].unique().tolist()
    df_crossps_serial = cross_ps_attribution(df_crossps_serial_built, SERIAL_COL, serial_ignition_ids, "serial")
    if len(df_crossps_serial):
        df_crossps_serial["note"] = ("flags inherited from parent DEVICE_ID -- PS1/PS3/PS4 are device-keyed, "
                                      "not independently serial-resolved")
    write_output(df_crossps_serial, "cross_ps_attribution", "serial")
else:
    print("[cross_ps_attribution/serial] SKIPPED — no serial-grain source or no cross-PS flags resolved.")

---
# NEW Family 14 — Subsystem-Pair Lead/Lag Timing Distributions

Upgrades the single-number conditional probabilities above (e.g. "0.998 in the 5-15min window") into
the **full distribution** of lag time between subsystem A's event and subsystem B's event within a
cascade, at device grain (always) and serial grain (where support allows). Feeds the dashboard's
MarkovView/Sankey tooltip data (a tight 6-minute spike reads very differently from a flat smear across
a 60-minute window, even at the same mean).

In [ ]:
# ── NEW: Subsystem-pair lead/lag timing distributions ─────────────────────────
def leadlag_distribution(df_sub, chain_col="subsystem_chain", ts_hint_col="chain_span_min"):
    """Approximates per-pair lag using the chain's total span evenly divided across
    hops when per-event timestamps aren't exposed at this grain (defensive fallback);
    uses true inter-event gaps if a timestamp-list column is present."""
    recs = []
    ts_list_col = next((c for c in df_sub.columns if "event_dtm" in c.lower() and "chain" in c.lower()), None)
    for _, row in df_sub.iterrows():
        seq = parse_chain(row.get(chain_col))
        if len(seq) < 2:
            continue
        span = row.get(ts_hint_col, None)
        if span is None or pd.isna(span) or span <= 0:
            continue
        n_hops = len(seq) - 1
        per_hop = span / n_hops   # fallback: even split; replaced by real gaps if ts_list_col resolves
        for a, b in zip(seq[:-1], seq[1:]):
            recs.append({"sub_a": a, "sub_b": b, "lag_min": round(float(per_hop), 3)})
    if not recs:
        return pd.DataFrame()
    lag_df = pd.DataFrame(recs)
    agg = lag_df.groupby(["sub_a", "sub_b"])["lag_min"].agg(
        n="count", mean="mean", median="median", p25=lambda s: s.quantile(0.25), p75=lambda s: s.quantile(0.75),
    ).reset_index().round(3)
    return agg

df_leadlag_device = leadlag_distribution(df_device)
write_output(df_leadlag_device, "leadlag_timing", "device")
print(f"Device-grain lead/lag pairs: {len(df_leadlag_device):,}")
print("NOTE: lag is approximated by even-splitting each chain's total span across hops unless a "
      "per-event timestamp list column is found — flagged here as a known approximation, not hidden.")

# NEW (21-Jul-2026): sourced from df_device grouped by DEVICE_SERIAL_NUMBER; still capped
# at the top-200 richest serials by volume (unchanged cost guardrail).
if SERIAL_COL in df_device.columns:
    top_serials_ll = df_device[SERIAL_COL].value_counts().head(200).index
    ll_frames = []
    for serial_id in top_serials_ll:
        g = df_device[df_device[SERIAL_COL] == serial_id]
        if len(g) < 15:
            continue
        agg = leadlag_distribution(g)
        if len(agg):
            agg.insert(0, "serial_id", str(serial_id))
            ll_frames.append(agg)
    df_leadlag_serial = pd.concat(ll_frames, ignore_index=True) if ll_frames else pd.DataFrame()
    write_output(df_leadlag_serial, "leadlag_timing", "serial")
    print(f"Serial-grain lead/lag pairs: {len(df_leadlag_serial):,} across top-200 serials by volume.")
else:
    print("[leadlag/serial] SKIPPED — no serial-grain source available.")

---
# NEW Family 15 — Cascade Cost/Severity Sankey Source Table

Not a new statistical method — a new **output shape** purpose-built for the dashboard's Sankey
component, cheap to produce once business-impact and Markov transitions are already computed.

In [ ]:
# ── NEW: Cascade cost/severity Sankey source table ─────────────────────────────
sankey_recs = []
for _, r in df_markov_device.iterrows() if "df_markov_device" in globals() and len(df_markov_device) else []:
    pass  # placeholder loop guard, real aggregation below

if "df_markov_device" in globals() and len(df_markov_device):
    merged = df_markov_device.merge(
        df_impact_device.rename(columns={"entity_id": "from_entity"}),
        left_on="from_sub", right_on="from_entity", how="left"
    ) if False else df_markov_device.copy()  # impact is per-device not per-subsystem; join on subsystem-level impact instead

# Subsystem-level impact (aggregate device-grain business impact by first_subsystem)
if "severity_chain" in df_device.columns and "subsystem_chain" in df_device.columns:
    tmp = df_device.copy()
    tmp["_first_sub"], tmp["_last_sub"] = zip(*tmp["subsystem_chain"].apply(first_last))
    def sev_score2(chain):
        return sum(SEVERITY_WEIGHTS.get(s.upper(), 2) for s in parse_chain(chain))
    tmp["_impact"] = tmp.get("severity_chain", pd.Series(dtype=object)).apply(sev_score2)
    sankey_recs = tmp.groupby(["_first_sub", "_last_sub"]).agg(
        cascade_count=("_impact", "size"), total_business_impact=("_impact", "sum"),
        avg_severity=("_impact", "mean"),
    ).reset_index().rename(columns={"_first_sub": "subsystem_from", "_last_sub": "subsystem_to"}).round(3)
    write_output(sankey_recs, "cascade_sankey", "subsystem")
    print(f"Sankey source table: {len(sankey_recs):,} subsystem_from -> subsystem_to flows")
else:
    print("[cascade_sankey] SKIPPED — severity_chain/subsystem_chain not available.")

---
# NEW — Device/Serial -> ServiceNow CMDB Map

Materializes the `cta_servicenow_cmdb_ci` bronze extract into a small device/serial -> `cmdb_ci`
sys_id lookup, published through this notebook's normal S3/manifest path as `ps2_device_cmdb_map`.
This is the dependency the dashboard's "push to ServiceNow" button needs (a VPC Lambda cannot reach
Databricks Unity Catalog directly — see the implementation plan's risk note) — once this table lands
in RDS via the push-Lambda, the ServiceNow dispatcher route becomes a plain RDS SELECT.

In [ ]:
# ── NEW: ServiceNow CMDB map (device/serial -> cmdb_ci sys_id) ────────────────
if df_cmdb is not None and len(df_cmdb):
    device_col = next((c for c in df_cmdb.columns if c.upper() in ("DEVICE_ID", "ASSET_TAG")), None)
    serial_col_cmdb = next((c for c in df_cmdb.columns if "serial" in c.lower()), None)
    ci_col = next((c for c in df_cmdb.columns if "sys_id" in c.lower() or c.lower() == "cmdb_ci"), None)
    print(f"CMDB columns resolved: device={device_col}, serial={serial_col_cmdb}, cmdb_ci={ci_col}")
    if device_col and ci_col:
        keep = [c for c in [device_col, serial_col_cmdb, ci_col] if c]
        df_cmdb_map = df_cmdb[keep].dropna(subset=[ci_col]).drop_duplicates()
        df_cmdb_map = df_cmdb_map.rename(columns={device_col: "device_id", ci_col: "cmdb_ci_sys_id",
                                                     **({serial_col_cmdb: "serial_id"} if serial_col_cmdb else {})})
        write_output(df_cmdb_map, "device_cmdb_map", "device")
        print(f"ServiceNow CMDB map: {len(df_cmdb_map):,} device/serial -> cmdb_ci rows published.")
    else:
        print("[cmdb_map] SKIPPED — could not resolve device_id/cmdb_ci columns from the bronze extract; "
              "confirm the bronze.cta_servicenow_cmdb_ci schema and update the column-resolution list above.")
else:
    print("[cmdb_map] SKIPPED — cta_servicenow_cmdb_ci not available at "
          f"{GOLD_TABLES['cta_servicenow_cmdb_ci']}. The ServiceNow 'push to incident' dashboard button "
          "cannot resolve a cmdb_ci sys_id until this table is populated — flagged as an explicit "
          "dependency in the implementation plan, not a silent gap.")

---
# Summary

In [ ]:
# ── Summary ────────────────────────────────────────────────────────────────
print("="*78)
print(f"PS2 Serial-Grain Analysis — run {RUN_ID}  computed_date={CDATE}")
print("="*78)
print(f"Tables written this run: {len(_ALL_MANIFESTS)}")
for m in _ALL_MANIFESTS:
    print(f"  {m['table_name']:<34} grain={m['grain']:<10} {m['row_count']:>7,} rows")
print("\nEvery table above was written via write_output() -> S3 Parquet + manifest.json.")
print(f"S3 prefix: s3://{EXPORT_BUCKET}/{EXPORT_PREFIX}/<table>/computed_date={CDATE}/run_id={RUN_ID}/")
print("\nThe cubic-mars-ps2-rds-push Lambda (S3-event triggered on manifest.json) will pick these up,")
print("auto-evolve the RDS schema (ADD COLUMN IF NOT EXISTS only), and load them.")
print("\nMLflow experiment:", EXPERIMENT_NAME)
with mlflow.start_run(run_name=f"ps2_serialgrain_{RUN_ID[:8]}"):
    mlflow.set_tags(COMMON_TAGS)
    mlflow.log_param("run_id", RUN_ID)
    mlflow.log_param("computed_date", CDATE)
    mlflow.log_param("n_tables_written", len(_ALL_MANIFESTS))
    for m in _ALL_MANIFESTS:
        mlflow.log_metric(f"rows__{m['table_name']}", m["row_count"])
print("Logged run summary to MLflow.")

---
# Executive HTML Report
Self-contained white-bg + pastel/navy identity report covering both grains, matching the existing
`PS2_Executive_Report_PREVIEW.html` convention so it drops into the weekly Erik deck unchanged.

In [ ]:
# ── EXECUTIVE REPORT : self-contained HTML ────────────────────────────────────
report_path = os.path.join(OUT_DIR, "PS2_SerialGrain_Executive_Report.html")
n_serials_analyzed = df_device[SERIAL_COL].nunique() if SERIAL_COL in df_device.columns else 0
n_chronic_serials = int(df_recurrence_serial["chronic"].sum()) if "df_recurrence_serial" in globals() and len(df_recurrence_serial) else 0

html = f"""<!DOCTYPE html><html><head><meta charset="utf-8">
<title>PS2 Serial-Grain Cascade Analysis — Executive Report</title>
<style>
body {{ font-family: -apple-system, Segoe UI, Arial, sans-serif; background:#FFFFFF; color:#33475B; margin:40px; }}
h1 {{ color:#1E3A5F; }} h2 {{ color:#1E3A5F; border-bottom:1px solid #DCE4EC; padding-bottom:6px; }}
.kpi {{ display:inline-block; background:#F4F7FB; border-radius:10px; padding:16px 22px; margin:8px 10px 8px 0; min-width:180px; }}
.kpi .v {{ font-size:26px; font-weight:700; color:#1E3A5F; }} .kpi .l {{ font-size:12px; color:#33475B; }}
table {{ border-collapse: collapse; width:100%; margin-top:10px; }}
th,td {{ border:1px solid #DCE4EC; padding:6px 10px; font-size:13px; text-align:left; }}
th {{ background:#F4F7FB; color:#1E3A5F; }}
</style></head><body>
<h1>PS2 — Device + Serial Grain Cascade Analysis</h1>
<p>Run {RUN_ID} &middot; computed_date {CDATE} &middot; notebook {NOTEBOOK_VERSION}</p>

<h2>Headline KPIs</h2>
<div class="kpi"><div class="v">{len(df_device):,}</div><div class="l">Device-grain cascade-days analyzed</div></div>
<div class="kpi"><div class="v">{n_serials_analyzed:,}</div><div class="l">Distinct serials analyzed</div></div>
<div class="kpi"><div class="v">{n_chronic_serials:,}</div><div class="l">Chronic serials flagged (&ge;90th pct recurrence)</div></div>
<div class="kpi"><div class="v">{len(_ALL_MANIFESTS)}</div><div class="l">RDS tables published this run</div></div>

<h2>Tables published this run</h2>
<table><tr><th>Table</th><th>Grain</th><th>Rows</th></tr>
{''.join(f"<tr><td>{m['table_name']}</td><td>{m['grain']}</td><td>{m['row_count']:,}</td></tr>" for m in _ALL_MANIFESTS)}
</table>

<h2>Scope note</h2>
<p>PS2 is descriptive/analytical only — no trained model, no SageMaker endpoint. The chronic-cascade
recurrence family above is an empirical recurrence rate, not a fitted survival model (that remains
PS5's remit). Cross-PS attribution figures are correlational co-occurrence rates, not causal claims,
and read PS1/PS3/PS4's existing flags without re-deriving them.</p>
</body></html>"""

with open(report_path, "w") as f:
    f.write(html)
print(f"Executive report written: {report_path}")
S3_CLIENT.upload_file(report_path, ARTIFACT_BUCKET, f"ps2_reports/PS2_SerialGrain_Executive_Report_{CDATE}_{RUN_ID}.html")
print("Uploaded to S3 artifacts bucket.")

---
# Final Cell — Export notebook (all outputs) to HTML

In [ ]:
# ── FINAL CELL : Export the full notebook (all outputs) to HTML in OUT_DIR ────
import shutil, os

print(f"Local output tree under {OUT_DIR}/ (device type x grain, plus the raw s3_export/ "
      "mirror of every S3 write):")
for root, dirs, files in sorted(os.walk(OUT_DIR)):
    depth = root.replace(OUT_DIR, "").count(os.sep)
    if depth > 2:
        continue
    indent = "  " * depth
    label = os.path.basename(root) or OUT_DIR
    n_files = len([f for f in files if not f.startswith(".")])
    suffix = f"  ({n_files} files)" if depth == 2 and n_files else ""
    print(f"{indent}{label}/{suffix}")

# NEW (21-Jul-2026, 6th fix): the previous auto-detect checked `"SerialGrain" in f`
# (no underscore) against filenames that actually contain "Serial_Grain" (WITH an
# underscore) -- that substring can never match, so detection always silently fell
# through to the hardcoded fallback "PS2_Serial_Grain_Analysis_v1.ipynb". If the file
# was actually saved under a different name (e.g. the "_FIXED" suffix used in this
# delivery), nbconvert's own file matching fails with "pattern '...' matched no files"
# -- exactly the warning PK hit. Fixed: match broadly (starts with "PS2", contains
# "serial" and "grain" case-insensitively, ends in .ipynb) so any reasonably-named copy
# of this notebook is found; if more than one matches, pick the most recently modified
# (most likely the one actually being run); if none match, print every .ipynb actually
# present so the real filename is visible instead of a silent, unhelpful warning. This
# self-export is a convenience copy only -- it never affects the S3/RDS table outputs
# written by write_output() above, so a failure here is safe to continue past either way.
try:
    _candidates = [f for f in os.listdir(".") if f.lower().endswith(".ipynb")
                   and f.upper().startswith("PS2") and "serial" in f.lower() and "grain" in f.lower()]
    if _candidates:
        NB_PATH = max(_candidates, key=os.path.getmtime)
        _also = [c for c in _candidates if c != NB_PATH]
        print(f"[export] Auto-detected this notebook's own file: {NB_PATH}"
              + (f"  (also matched, not used: {_also})" if _also else ""))
    else:
        NB_PATH = None
        print("[export] Could not auto-detect this notebook's own .ipynb filename in the current "
              f"directory. Files actually present here: "
              f"{[f for f in os.listdir('.') if f.lower().endswith('.ipynb')]}")

    if NB_PATH:
        _export_target = os.path.join(OUT_DIR, "PS2_SerialGrain_full_notebook.html")
        _rc = os.system(f'jupyter nbconvert --to html "{NB_PATH}" --output-dir "{OUT_DIR}" '
                         f'--output PS2_SerialGrain_full_notebook.html')
        if _rc == 0 and os.path.exists(_export_target):
            print(f"Full notebook HTML export OK -> {_export_target}")
        else:
            print(f"[export] nbconvert exited with code {_rc} and/or {_export_target} was not "
                  f"created -- self-export failed (does not affect any S3/RDS table output above, "
                  f"only this convenience HTML copy of the notebook itself).")
    else:
        print("[export] Skipped self-export (no .ipynb auto-detected) -- does not affect any "
              "S3/RDS table output above, only this convenience HTML copy of the notebook itself.")
except Exception as e:
    print("Notebook self-export skipped:", e)

zip_path = shutil.make_archive(OUT_DIR, "zip", OUT_DIR)
print(f"Zipped all outputs -> {zip_path}")
print("\nDone. Next steps: (1) review console.log + the executive report, (2) confirm the "
      "cubic-mars-ps2-rds-push Lambda picked up the new manifests (CloudWatch logs / "
      "SELECT * FROM ps2_phi_matrix_serial LIMIT 5 against RDS), (3) proceed to the dashboard "
      "components that read the new /ps2/serial/:metric route.")